# Cicero full agent bootstrap planner observability

This notebook belongs to the project's sequential measurement and validation programme. Read its result as evidence about behavioural validity, representation, comparator strength, timing, information matching, or mechanistic calibration as appropriate. Legacy H1/H_priv identifiers may remain inside code, saved paths, or frozen schemas for reproducibility; they are not the object being "found" by the current analysis.

**Repository framing.** The current paper separates target-specific representation from predictive privilege. A positive neural result is interpreted only after behavioural validity, control, comparator-strength, timing, and information-set checks.


# Latent reservations — Notebook 14

Released CICERO full-agent bootstrap and strategic planner observability.

### v5.7 source-built Python runtime

This revision removes the managed Python download path.

- CPython 3.7.17 is built from the official source release into a project-local prefix.
- OpenSSL 1.1.1w is built into a project-local prefix and linked explicitly into CPython.
- A standard Python virtual environment is then created from that interpreter.
- Python dependencies are installed with the virtual environment's `pip`.
- PyTorch 1.7.1 uses the released CUDA 11.0 wheel.
- The earlier legacy Python and compiler prefixes are not reused.


### Selective released-weight storage path

The released all-model download path is not invoked. The notebook resolves the concrete CICERO
configuration, records HeyHi config-group mounts without attempting to read directories as files,
maps reachable model references to the released manifest, probes sizes before transfer, and
downloads/decrypts one selected file at a time while immediately removing its encrypted staging copy.


In [1]:
NOTEBOOK_BUILD = "latent-reservations-notebook14-cicero-full-agent-bootstrap-v5.7-source-py37-venv"
print("=" * 100)
print(f"NOTEBOOK BUILD: {NOTEBOOK_BUILD}")
print("CICERO full-agent bootstrap + planner observability audit")
print("=" * 100)


NOTEBOOK BUILD: latent-reservations-notebook14-cicero-full-agent-bootstrap-v5.7-source-py37-venv
CICERO full-agent bootstrap + planner observability audit


## 1. Paths, provenance, and outputs


In [2]:
import hashlib
import json
import os
import re
import shutil
import subprocess
import sys
import threading
import queue
import time
from datetime import datetime, timezone
from pathlib import Path

try:
    from tqdm.auto import tqdm
except Exception:
    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "tqdm>=4.66",
        ],
        check=True,
    )
    from tqdm.auto import tqdm

PROJECT_ROOT = Path(
    os.environ.get(
        "LR_PROJECT_ROOT",
        "/workspace/latent-reservations",
    )
).expanduser().resolve()

NOTEBOOK_SLUG = "14_cicero_full_agent_bootstrap_planner_observability"

RUN_ID = os.environ.get(
    "LR_RUN_ID",
    datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ"),
)

BASE = PROJECT_ROOT / "notebook_outputs" / NOTEBOOK_SLUG
RUN_DIR = BASE / RUN_ID
MANIFEST_DIR = RUN_DIR / "manifests"
DATA_DIR = RUN_DIR / "data"
ENGINE_DIR = DATA_DIR / "engine"
SOURCE_DIR = DATA_DIR / "source_audit"
RUNTIME_DIR = DATA_DIR / "runtime"
TABLE_DIR = RUN_DIR / "results" / "tables"

for path in [
    MANIFEST_DIR,
    ENGINE_DIR,
    SOURCE_DIR,
    RUNTIME_DIR,
    TABLE_DIR,
]:
    path.mkdir(
        parents=True,
        exist_ok=True,
    )

BASE.mkdir(
    parents=True,
    exist_ok=True,
)

(BASE / "latest_run.json").write_text(
    json.dumps(
        {
            "notebook_slug": NOTEBOOK_SLUG,
            "run_id": RUN_ID,
            "run_output_dir": str(RUN_DIR),
            "updated_at_utc": datetime.now(timezone.utc).isoformat(),
        },
        indent=2,
    )
)

CICERO_REPO = (
    PROJECT_ROOT
    / "vendor"
    / "diplomacy_cicero"
)

NB11_BASE = (
    PROJECT_ROOT
    / "notebook_outputs"
    / "11_cicero_native_engine_legal_order_replay"
)

NB11_RUN_DIR = Path(
    json.loads(
        (NB11_BASE / "latest_run.json").read_text()
    )[
        "run_output_dir"
    ]
)

nb11_summary = json.loads(
    (
        NB11_RUN_DIR
        / "results"
        / "tables"
        / "result_summary.json"
    ).read_text()
)

CICERO_COMMIT = nb11_summary[
    "cicero_commit"
]

if not CICERO_REPO.exists():
    raise FileNotFoundError(
        f"Frozen CICERO checkout not found: {CICERO_REPO}"
    )

actual_commit = subprocess.check_output(
    [
        "git",
        "rev-parse",
        "HEAD",
    ],
    cwd=CICERO_REPO,
    text=True,
).strip()

if actual_commit != CICERO_COMMIT:
    subprocess.run(
        [
            "git",
            "checkout",
            "--detach",
            CICERO_COMMIT,
        ],
        cwd=CICERO_REPO,
        check=True,
    )

tracked_status = subprocess.check_output(
    [
        "git",
        "status",
        "--porcelain",
        "--untracked-files=no",
    ],
    cwd=CICERO_REPO,
    text=True,
).strip()

if tracked_status:
    raise RuntimeError(
        "Frozen CICERO checkout contains tracked modifications."
    )

print({
    "run_dir": str(RUN_DIR),
    "cicero_repo": str(CICERO_REPO),
    "cicero_commit": CICERO_COMMIT,
})


{'run_dir': '/workspace/latent-reservations/notebook_outputs/14_cicero_full_agent_bootstrap_planner_observability/20260815T122832Z', 'cicero_repo': '/workspace/latent-reservations/vendor/diplomacy_cicero', 'cicero_commit': 'e85afeddb34f5b7c1ea0827203b425a0f7e68ead'}


## 2. Hardware and host toolchain inventory


In [3]:
def capture_command(
    command,
    *,
    cwd=None,
    env=None,
):
    process = subprocess.run(
        [
            str(
                item
            )
            for item in command
        ],
        cwd=cwd,
        env=env,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        check=False,
    )

    return {
        "command": [
            str(
                item
            )
            for item in command
        ],
        "returncode": int(
            process.returncode
        ),
        "output": process.stdout,
    }


host_inventory = {
    "python_executable": sys.executable,
    "python_version": sys.version,
    "platform": sys.platform,
    "nvidia_smi": capture_command(
        [
            "nvidia-smi",
        ]
    ),
    "gcc": capture_command(
        [
            "gcc",
            "--version",
        ]
    ),
    "gxx": capture_command(
        [
            "g++",
            "--version",
        ]
    ),
    "cmake": capture_command(
        [
            "cmake",
            "--version",
        ]
    ),
    "make": capture_command(
        [
            "make",
            "--version",
        ]
    ),
    "go": capture_command(
        [
            "bash",
            "-c",
            "command -v go && go version",
        ]
    ),
    "nvcc": capture_command(
        [
            "bash",
            "-c",
            "command -v nvcc && nvcc --version",
        ]
    ),
    "git": capture_command(
        [
            "git",
            "--version",
        ]
    ),
}

(
    ENGINE_DIR
    / "host_inventory.json"
).write_text(
    json.dumps(
        host_inventory,
        indent=2,
    )
)

print(
    json.dumps(
        {
            key: (
                value[
                    "returncode"
                ]
                if isinstance(
                    value,
                    dict,
                )
                and "returncode"
                in value
                else value
            )
            for key, value in host_inventory.items()
        },
        indent=2,
    )
)


{
  "python_executable": "/usr/local/bin/python",
  "python_version": "3.12.3 (main, Aug 14 2025, 17:47:21) [GCC 13.3.0]",
  "platform": "linux",
  "nvidia_smi": 0,
  "gcc": 0,
  "gxx": 0,
  "cmake": 0,
  "make": 0,
  "go": 1,
  "nvcc": 0,
  "git": 0
}


## 3. Static source audit of CICERO's strategic path


In [4]:
SOURCE_PATTERNS = {
    "bqre_agent_class": re.compile(
        r"class\s+BQRE1PAgent\b"
    ),
    "run_search": re.compile(
        r"def\s+run_search\s*\("
    ),
    "blueprint_policy": re.compile(
        r"\bbp_policy\b|\bblueprint\b",
        flags=re.IGNORECASE,
    ),
    "plausible_orders": re.compile(
        r"plausible_orders|PlausibleOrders"
    ),
    "action_probabilities": re.compile(
        r"power_action_ps|action_ps|get_cur_iter_strategies"
    ),
    "belief_state": re.compile(
        r"belief_state|BeliefState"
    ),
    "bilateral": re.compile(
        r"bilateral|Bilateral",
    ),
    "dialogue": re.compile(
        r"dialogue|Dialogue|message",
    ),
}

source_hits = []

python_files = sorted(
    (
        CICERO_REPO
        / "fairdiplomacy"
    ).rglob(
        "*.py"
    )
)

for path in tqdm(
    python_files,
    desc="Audit CICERO planner source",
    unit="file",
):
    try:
        lines = path.read_text(
            errors="replace"
        ).splitlines()
    except Exception:
        continue

    for line_number, line in enumerate(
        lines,
        start=1,
    ):
        matched = [
            name
            for name, pattern in SOURCE_PATTERNS.items()
            if pattern.search(
                line
            )
        ]

        if matched:
            source_hits.append({
                "path": str(
                    path.relative_to(
                        CICERO_REPO
                    )
                ),
                "line": int(
                    line_number
                ),
                "patterns": matched,
                "text": line.strip(),
            })

(
    SOURCE_DIR
    / "planner_source_hits.json"
).write_text(
    json.dumps(
        source_hits,
        indent=2,
    )
)

required_source_signals = {
    "BQRE1PAgent": any(
        "bqre_agent_class"
        in row[
            "patterns"
        ]
        for row in source_hits
    ),
    "run_search": any(
        "run_search"
        in row[
            "patterns"
        ]
        for row in source_hits
    ),
    "blueprint_policy": any(
        "blueprint_policy"
        in row[
            "patterns"
        ]
        for row in source_hits
    ),
    "plausible_orders": any(
        "plausible_orders"
        in row[
            "patterns"
        ]
        for row in source_hits
    ),
    "action_probabilities": any(
        "action_probabilities"
        in row[
            "patterns"
        ]
        for row in source_hits
    ),
    "belief_state": any(
        "belief_state"
        in row[
            "patterns"
        ]
        for row in source_hits
    ),
    "bilateral": any(
        "bilateral"
        in row[
            "patterns"
        ]
        for row in source_hits
    ),
}

(
    TABLE_DIR
    / "planner_source_signals.json"
).write_text(
    json.dumps(
        required_source_signals,
        indent=2,
    )
)

print(
    json.dumps(
        required_source_signals,
        indent=2,
    )
)


Audit CICERO planner source:   0%|          | 0/137 [00:00<?, ?file/s]

{
  "BQRE1PAgent": true,
  "run_search": true,
  "blueprint_policy": true,
  "plausible_orders": true,
  "action_probabilities": true,
  "belief_state": true,
  "bilateral": true
}


## 4. Freeze released CICERO agent configuration and model references


In [5]:
CICERO_AGENT_INCLUDE = "agents/cicero.prototxt"
CICERO_IMITATION_AGENT_INCLUDE = "agents/ablations/cicero_imitation_only.prototxt"

COMPARE_AGENTS_CONFIG = (
    CICERO_REPO
    / "conf"
    / "c01_ag_cmp"
    / "cmp.prototxt"
)

if not COMPARE_AGENTS_CONFIG.exists():
    raise FileNotFoundError(
        COMPARE_AGENTS_CONFIG
    )

compare_config_text = COMPARE_AGENTS_CONFIG.read_text(
    errors="replace"
)

# HeyHi include identifiers are logical config references.  The checkout may
# or may not expose a backing file at a path that directly mirrors the
# include string, so inventory candidate backing files rather than requiring
# one guessed location.
backing_agent_config_candidates = sorted(
    path
    for path in (
        CICERO_REPO
        / "conf"
    ).rglob(
        "*.prototxt"
    )
    if (
        path.name.lower()
        == "cicero.prototxt"
        or "cicero"
        in path.name.lower()
    )
)

backing_agent_config_rows = []

for path in backing_agent_config_candidates:
    try:
        payload = path.read_text(
            errors="replace"
        )
    except Exception:
        payload = ""

    backing_agent_config_rows.append({
        "path": str(
            path.relative_to(
                CICERO_REPO
            )
        ),
        "sha256": hashlib.sha256(
            path.read_bytes()
        ).hexdigest(),
        "size_bytes": int(
            path.stat().st_size
        ),
        "preview": payload[
            :2000
        ],
    })

# The released compare task itself proves that agent configs are mounted
# through include identifiers.
compare_include_lines = [
    line.strip()
    for line in compare_config_text.splitlines()
    if "includes" in line
]

if not compare_include_lines:
    raise RuntimeError(
        "Released compare-agents config does not contain HeyHi include declarations."
    )

config_snapshot = {
    "compare_agents_config": str(
        COMPARE_AGENTS_CONFIG
    ),
    "compare_agents_config_sha256": hashlib.sha256(
        COMPARE_AGENTS_CONFIG.read_bytes()
    ).hexdigest(),
    "compare_agents_config_text": compare_config_text,
    "compare_include_lines": compare_include_lines,
    "cicero_agent_include": CICERO_AGENT_INCLUDE,
    "cicero_imitation_agent_include": CICERO_IMITATION_AGENT_INCLUDE,
    "backing_agent_config_candidates": backing_agent_config_rows,
    "path_semantics": (
        "CICERO agent override is a HeyHi logical include identifier, "
        "not a repository-root filesystem path."
    ),
}

(
    SOURCE_DIR
    / "released_agent_config_snapshot.json"
).write_text(
    json.dumps(
        config_snapshot,
        indent=2,
    )
)

print({
    "compare_agents_config": str(
        COMPARE_AGENTS_CONFIG
    ),
    "cicero_agent_include": CICERO_AGENT_INCLUDE,
    "backing_config_candidates": len(
        backing_agent_config_rows
    ),
    "compare_include_lines": compare_include_lines,
})


{'compare_agents_config': '/workspace/latent-reservations/vendor/diplomacy_cicero/conf/c01_ag_cmp/cmp.prototxt', 'cicero_agent_include': 'agents/cicero.prototxt', 'backing_config_candidates': 5, 'compare_include_lines': ['includes {path: "agents/searchbot"; mount: "agent_one";}', 'includes {path: "agents/base_strategy_model"; mount: "agent_six";}']}


## 5. Ensure released pretrained model files are present


In [6]:
MODEL_DIR = (
    CICERO_REPO
    / "models"
)

ENCRYPTED_DIR = (
    CICERO_REPO
    / "models_encrypted"
)

RELEASE_MANIFEST = (
    CICERO_REPO
    / "bin"
    / "s3_filenames.txt"
)

MODEL_BASE_URL = (
    "https://dl.fbaipublicfiles.com/diplomacy_cicero/models"
)

OFFICIAL_MODEL_PASSWORD = os.environ.get(
    "CICERO_MODEL_PASSWORD",
    "dbEmG*yo@fuWzb79cx_pN7.TRm4cqk",
)

ENVIRONMENT_RESERVE_GIB = float(
    os.environ.get(
        "LR_14_ENV_RESERVE_GIB",
        "8",
    )
)

WORKING_RESERVE_GIB = float(
    os.environ.get(
        "LR_14_WORKING_RESERVE_GIB",
        "2",
    )
)

GIB = 1024 ** 3

if not RELEASE_MANIFEST.exists():
    raise FileNotFoundError(
        RELEASE_MANIFEST
    )

if shutil.which(
    "gpg"
) is None:
    raise RuntimeError(
        "gpg is required to decrypt the released model files."
    )

all_configs = sorted(
    path
    for path in (
        CICERO_REPO
        / "conf"
    ).rglob(
        "*.prototxt"
    )
    if path.is_file()
)


def include_variants(
    identifier,
):
    identifier = str(
        identifier
    ).strip().lstrip(
        "./"
    )

    variants = [
        identifier
    ]

    if not identifier.endswith(
        ".prototxt"
    ):
        variants.append(
            identifier
            + ".prototxt"
        )

    return list(
        dict.fromkeys(
            variants
        )
    )


def resolve_include_file(
    identifier,
):
    variants = include_variants(
        identifier
    )

    for variant in variants:
        for root in [
            CICERO_REPO
            / "conf",
            CICERO_REPO
            / "conf"
            / "common",
        ]:
            candidate = (
                root
                / variant
            )

            if (
                candidate.exists()
                and candidate.is_file()
            ):
                return candidate.resolve()

    matches = []

    for path in all_configs:
        relative = str(
            path.relative_to(
                CICERO_REPO
                / "conf"
            )
        ).replace(
            "\\",
            "/",
        )

        if any(
            relative.endswith(
                variant
            )
            for variant in variants
        ):
            matches.append(
                path.resolve()
            )

    matches = list(
        dict.fromkeys(
            matches
        )
    )

    if len(
        matches
    ) == 1:
        return matches[
            0
        ]

    basenames = {
        Path(
            variant
        ).name
        for variant in variants
    }

    basename_matches = list(
        dict.fromkeys(
            path.resolve()
            for path in all_configs
            if path.name in basenames
        )
    )

    if len(
        basename_matches
    ) == 1:
        return basename_matches[
            0
        ]

    return None


def resolve_include_group(
    identifier,
):
    identifier = str(
        identifier
    ).strip().lstrip(
        "./"
    )

    if identifier.endswith(
        ".prototxt"
    ):
        return None

    for root in [
        CICERO_REPO
        / "conf",
        CICERO_REPO
        / "conf"
        / "common",
    ]:
        candidate = (
            root
            / identifier
        )

        if (
            candidate.exists()
            and candidate.is_dir()
        ):
            return candidate.resolve()

    return None


include_block_re = re.compile(
    r"includes\s*\{(.*?)\}",
    flags=re.DOTALL,
)

include_path_re = re.compile(
    r'path\s*:\s*"([^"]+)"'
)

model_path_re = re.compile(
    r'(?:\$\{[^}]+\}/|[A-Za-z0-9_./-]*/)?models/([A-Za-z0-9_./-]+)'
)


def includes_from_text(
    text,
):
    values = []

    for block in include_block_re.findall(
        text
    ):
        values.extend(
            include_path_re.findall(
                block
            )
        )

    return list(
        dict.fromkeys(
            value.strip()
            for value in values
            if value.strip()
        )
    )


def model_literals_from_text(
    text,
):
    values = []

    for match in model_path_re.finditer(
        text
    ):
        value = (
            match.group(
                1
            )
            .strip()
            .rstrip(
                '",}]'
            )
            .strip(
                "/"
            )
        )

        if value:
            values.append(
                value
            )

    return list(
        dict.fromkeys(
            values
        )
    )


cicero_agent_config = resolve_include_file(
    CICERO_AGENT_INCLUDE
)

if cicero_agent_config is None:
    named_candidates = [
        str(
            path.relative_to(
                CICERO_REPO
            )
        )
        for path in all_configs
        if (
            path.is_file()
            and "cicero"
            in path.name.lower()
        )
    ]

    raise RuntimeError(
        "Could not resolve the concrete CICERO agent configuration. "
        f"Candidates: {named_candidates}"
    )

pending = [
    COMPARE_AGENTS_CONFIG.resolve(),
    cicero_agent_config.resolve(),
]

visited = set()
graph_rows = []
config_group_mounts = []
unresolved_includes = []
model_literals = []
visited_texts = {}

while pending:
    path = pending.pop(
        0
    ).resolve()

    if path in visited:
        continue

    if not path.is_file():
        raise RuntimeError(
            "A non-file entered the configuration traversal: "
            f"{path}"
        )

    visited.add(
        path
    )

    text = path.read_text(
        errors="replace"
    )

    visited_texts[
        str(
            path
        )
    ] = text

    direct_includes = includes_from_text(
        text
    )

    direct_models = model_literals_from_text(
        text
    )

    model_literals.extend(
        direct_models
    )

    children = []
    mounts = []

    for include_id in direct_includes:
        child_file = resolve_include_file(
            include_id
        )

        if child_file is not None:
            children.append(
                str(
                    child_file.relative_to(
                        CICERO_REPO
                    )
                )
            )

            if child_file not in visited:
                pending.append(
                    child_file
                )

            continue

        group_dir = resolve_include_group(
            include_id
        )

        if group_dir is not None:
            record = {
                "source": str(
                    path.relative_to(
                        CICERO_REPO
                    )
                ),
                "include": include_id,
                "directory": str(
                    group_dir.relative_to(
                        CICERO_REPO
                    )
                ),
            }

            config_group_mounts.append(
                record
            )

            mounts.append(
                record
            )

            continue

        unresolved_includes.append({
            "source": str(
                path.relative_to(
                    CICERO_REPO
                )
            ),
            "include": include_id,
        })

    graph_rows.append({
        "path": str(
            path.relative_to(
                CICERO_REPO
            )
        ),
        "includes": direct_includes,
        "resolved_children": children,
        "config_group_mounts": mounts,
        "model_literals": direct_models,
    })

released_files = [
    line.strip()
    for line in RELEASE_MANIFEST.read_text().splitlines()
    if line.strip()
]

released_set = set(
    released_files
)

model_literals = sorted(
    set(
        model_literals
    )
)


def released_matches(
    literal,
):
    candidates = {
        literal,
        Path(
            literal
        ).name,
    }

    expanded = set(
        candidates
    )

    for candidate in list(
        candidates
    ):
        if candidate in released_set:
            for suffix in [
                ".dict",
                ".opt",
            ]:
                sidecar = (
                    candidate
                    + suffix
                )

                if sidecar in released_set:
                    expanded.add(
                        sidecar
                    )

    return sorted(
        candidate
        for candidate in expanded
        if candidate in released_set
    )


selected_release_files = {
    filename
    for literal in model_literals
    for filename in released_matches(
        literal
    )
}

combined_config_text = "\n".join(
    visited_texts.values()
)

for filename in released_files:
    if filename in combined_config_text:
        selected_release_files.add(
            filename
        )

        for suffix in [
            ".dict",
            ".opt",
        ]:
            sidecar = (
                filename
                + suffix
            )

            if sidecar in released_set:
                selected_release_files.add(
                    sidecar
                )

selected_release_files = sorted(
    selected_release_files
)

unmatched_model_literals = sorted(
    literal
    for literal in model_literals
    if not released_matches(
        literal
    )
)

if not selected_release_files:
    raise RuntimeError(
        "The resolved full-agent configuration did not map to any "
        "released model files."
    )

subset_payload = {
    "cicero_agent_include": CICERO_AGENT_INCLUDE,
    "resolved_cicero_agent_config": str(
        cicero_agent_config.relative_to(
            CICERO_REPO
        )
    ),
    "config_graph": graph_rows,
    "config_group_mounts": config_group_mounts,
    "unresolved_includes": unresolved_includes,
    "model_literals": model_literals,
    "selected_release_files": selected_release_files,
    "unmatched_model_literals": unmatched_model_literals,
    "released_manifest_count": len(
        released_files
    ),
    "selected_count": len(
        selected_release_files
    ),
}

(
    TABLE_DIR
    / "selected_model_subset.json"
).write_text(
    json.dumps(
        subset_payload,
        indent=2,
    )
)

print(
    json.dumps(
        {
            "resolved_agent_config": subset_payload[
                "resolved_cicero_agent_config"
            ],
            "concrete_config_files": len(
                visited
            ),
            "config_group_mounts": len(
                config_group_mounts
            ),
            "unresolved_includes": unresolved_includes,
            "selected_model_files": len(
                selected_release_files
            ),
        },
        indent=2,
    )
)

for filename in selected_release_files:
    print(
        "MODEL",
        filename,
    )


def directory_size_bytes(
    root,
):
    root = Path(
        root
    )

    if not root.exists():
        return 0

    total = 0

    for path in root.rglob(
        "*"
    ):
        try:
            if path.is_file():
                total += int(
                    path.stat().st_size
                )
        except FileNotFoundError:
            pass

    return total


def disk_snapshot():
    usage = shutil.disk_usage(
        PROJECT_ROOT
    )

    return {
        "total_bytes": int(
            usage.total
        ),
        "used_bytes": int(
            usage.used
        ),
        "free_bytes": int(
            usage.free
        ),
        "free_gib": float(
            usage.free
            / GIB
        ),
        "models_bytes": int(
            directory_size_bytes(
                MODEL_DIR
            )
        ),
        "models_encrypted_bytes": int(
            directory_size_bytes(
                ENCRYPTED_DIR
            )
        ),
    }


# Remove only encrypted staging payloads from interrupted earlier transfers.
# The decrypted model directory is never removed here.
if ENCRYPTED_DIR.exists():
    for path in sorted(
        ENCRYPTED_DIR.rglob(
            "*.gpg"
        )
    ):
        path.unlink(
            missing_ok=True
        )


def probe_remote_size(
    filename,
):
    url = (
        f"{MODEL_BASE_URL}/{filename}.gpg"
    )

    head = subprocess.run(
        [
            "curl",
            "-L",
            "--fail",
            "--silent",
            "--show-error",
            "--head",
            url,
        ],
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        check=False,
    )

    if head.returncode == 0:
        lengths = re.findall(
            r"(?im)^content-length:\s*(\d+)\s*$",
            head.stdout,
        )

        if lengths:
            return {
                "filename": filename,
                "url": url,
                "size_bytes": int(
                    lengths[
                        -1
                    ]
                ),
                "probe": "HEAD",
            }

    ranged = subprocess.run(
        [
            "curl",
            "-L",
            "--fail",
            "--silent",
            "--show-error",
            "--range",
            "0-0",
            "--dump-header",
            "-",
            "--output",
            "/dev/null",
            url,
        ],
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        check=False,
    )

    if ranged.returncode == 0:
        ranges = re.findall(
            r"(?im)^content-range:\s*bytes\s+\d+-\d+/(\d+)\s*$",
            ranged.stdout,
        )

        if ranges:
            return {
                "filename": filename,
                "url": url,
                "size_bytes": int(
                    ranges[
                        -1
                    ]
                ),
                "probe": "RANGE",
            }

    return {
        "filename": filename,
        "url": url,
        "size_bytes": None,
        "probe": "FAILED",
        "head_tail": head.stdout[
            -1000:
        ],
        "range_tail": ranged.stdout[
            -1000:
        ],
    }


MODEL_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

size_rows = []

for filename in tqdm(
    selected_release_files,
    desc="Probe selected model sizes",
    unit="file",
):
    local_path = (
        MODEL_DIR
        / filename
    )

    if local_path.exists():
        size_rows.append({
            "filename": filename,
            "url": (
                f"{MODEL_BASE_URL}/{filename}.gpg"
            ),
            "size_bytes": int(
                local_path.stat().st_size
            ),
            "probe": "LOCAL",
            "already_present": True,
        })
        continue

    row = probe_remote_size(
        filename
    )

    row[
        "already_present"
    ] = False

    size_rows.append(
        row
    )

(
    TABLE_DIR
    / "selected_model_remote_sizes.json"
).write_text(
    json.dumps(
        size_rows,
        indent=2,
    )
)

unknown_sizes = [
    row[
        "filename"
    ]
    for row in size_rows
    if row[
        "size_bytes"
    ]
    is None
]

if unknown_sizes:
    raise RuntimeError(
        "Remote size could not be established for every selected model. "
        "No model transfer was started. "
        f"Unknown: {unknown_sizes}"
    )

missing_rows = [
    row
    for row in size_rows
    if not row[
        "already_present"
    ]
]

download_bytes = int(
    sum(
        row[
            "size_bytes"
        ]
        for row in missing_rows
    )
)

existing_selected_bytes = int(
    sum(
        row[
            "size_bytes"
        ]
        for row in size_rows
        if row[
            "already_present"
        ]
    )
)

largest_encrypted_bytes = int(
    max(
        [
            row[
                "size_bytes"
            ]
            for row in missing_rows
        ]
        or [
            0
        ]
    )
)

estimated_new_decrypted_bytes = int(
    download_bytes
    * 1.03
)

estimated_final_selected_bytes = int(
    existing_selected_bytes
    + estimated_new_decrypted_bytes
)

estimated_peak_model_bytes = int(
    estimated_final_selected_bytes
    + largest_encrypted_bytes
)

before = disk_snapshot()

additional_peak_bytes = max(
    0,
    estimated_peak_model_bytes
    - before[
        "models_bytes"
    ],
)

required_free_bytes = int(
    additional_peak_bytes
    + (
        ENVIRONMENT_RESERVE_GIB
        + WORKING_RESERVE_GIB
    )
    * GIB
)

storage_plan = {
    "filesystem_before": before,
    "selected_file_count": len(
        size_rows
    ),
    "missing_file_count": len(
        missing_rows
    ),
    "download_bytes": download_bytes,
    "download_gib": float(
        download_bytes
        / GIB
    ),
    "estimated_final_selected_gib": float(
        estimated_final_selected_bytes
        / GIB
    ),
    "largest_encrypted_file_gib": float(
        largest_encrypted_bytes
        / GIB
    ),
    "environment_reserve_gib": ENVIRONMENT_RESERVE_GIB,
    "working_reserve_gib": WORKING_RESERVE_GIB,
    "required_free_gib": float(
        required_free_bytes
        / GIB
    ),
    "fits_current_filesystem": bool(
        required_free_bytes
        <= before[
            "free_bytes"
        ]
    ),
}

(
    TABLE_DIR
    / "storage_plan.json"
).write_text(
    json.dumps(
        storage_plan,
        indent=2,
    )
)

print(
    json.dumps(
        storage_plan,
        indent=2,
    )
)

if not storage_plan[
    "fits_current_filesystem"
]:
    raise RuntimeError(
        "The selected model subset does not fit with the configured "
        "runtime and working-space reserves. No model transfer was started."
    )

ENCRYPTED_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

transfer_rows = []

for row in tqdm(
    size_rows,
    desc="Fetch selected CICERO model subset",
    unit="file",
):
    filename = row[
        "filename"
    ]

    target = (
        MODEL_DIR
        / filename
    )

    target.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    if target.exists():
        transfer_rows.append({
            "filename": filename,
            "status": "already_present",
            "decrypted_size_bytes": int(
                target.stat().st_size
            ),
        })
        continue

    encrypted = (
        ENCRYPTED_DIR
        / f"{filename}.gpg"
    )

    encrypted.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    safe_name = re.sub(
        r"[^A-Za-z0-9_.-]+",
        "_",
        filename,
    )

    download_log = (
        ENGINE_DIR
        / f"download_{safe_name}.log"
    )

    url = (
        f"{MODEL_BASE_URL}/{filename}.gpg"
    )

    with download_log.open(
        "w",
        encoding="utf-8",
    ) as log_handle:
        download = subprocess.run(
            [
                "wget",
                "--continue",
                "--progress=dot:giga",
                url,
                "-O",
                str(
                    encrypted
                ),
            ],
            stdout=log_handle,
            stderr=subprocess.STDOUT,
            text=True,
            check=False,
        )

    if download.returncode != 0:
        raise RuntimeError(
            f"Model download failed for {filename}; see {download_log}"
        )

    expected_size = int(
        row[
            "size_bytes"
        ]
    )

    actual_size = int(
        encrypted.stat().st_size
    )

    if actual_size != expected_size:
        raise RuntimeError(
            f"Encrypted size mismatch for {filename}: "
            f"expected {expected_size}, got {actual_size}"
        )

    decrypt_log = (
        ENGINE_DIR
        / f"decrypt_{safe_name}.log"
    )

    with decrypt_log.open(
        "w",
        encoding="utf-8",
    ) as log_handle:
        decrypt = subprocess.run(
            [
                "gpg",
                "--batch",
                "--yes",
                "--pinentry-mode",
                "loopback",
                "--passphrase",
                OFFICIAL_MODEL_PASSWORD,
                "--output",
                str(
                    target
                ),
                "-d",
                str(
                    encrypted
                ),
            ],
            stdout=log_handle,
            stderr=subprocess.STDOUT,
            text=True,
            check=False,
        )

    if decrypt.returncode != 0:
        target.unlink(
            missing_ok=True
        )
        raise RuntimeError(
            f"Model decryption failed for {filename}; see {decrypt_log}"
        )

    decrypted_size = int(
        target.stat().st_size
    )

    encrypted.unlink(
        missing_ok=True
    )

    transfer_rows.append({
        "filename": filename,
        "status": "downloaded_decrypted_encrypted_removed",
        "encrypted_size_bytes": actual_size,
        "decrypted_size_bytes": decrypted_size,
    })

if ENCRYPTED_DIR.exists():
    for path in sorted(
        ENCRYPTED_DIR.rglob(
            "*"
        ),
        key=lambda value: len(
            value.parts
        ),
        reverse=True,
    ):
        if path.is_dir():
            try:
                path.rmdir()
            except OSError:
                pass

    try:
        ENCRYPTED_DIR.rmdir()
    except OSError:
        pass

missing_after = [
    filename
    for filename in selected_release_files
    if not (
        MODEL_DIR
        / filename
    ).exists()
]

if missing_after:
    raise RuntimeError(
        f"Selected model subset incomplete: {missing_after}"
    )

after_models = [
    {
        "path": str(
            (
                MODEL_DIR
                / filename
            ).relative_to(
                CICERO_REPO
            )
        ),
        "size_bytes": int(
            (
                MODEL_DIR
                / filename
            ).stat().st_size
        ),
    }
    for filename in selected_release_files
]

final_selected_bytes = int(
    sum(
        row[
            "size_bytes"
        ]
        for row in after_models
    )
)

after = disk_snapshot()

(
    ENGINE_DIR
    / "model_inventory.json"
).write_text(
    json.dumps(
        {
            "bulk_downloader_invoked": False,
            "selection_mode": "resolved concrete full-agent configuration",
            "resolved_agent_config": str(
                cicero_agent_config.relative_to(
                    CICERO_REPO
                )
            ),
            "config_group_mounts": config_group_mounts,
            "files": after_models,
            "transfers": transfer_rows,
            "total_bytes": final_selected_bytes,
            "total_gib": float(
                final_selected_bytes
                / GIB
            ),
            "encrypted_payload_bytes_retained": int(
                directory_size_bytes(
                    ENCRYPTED_DIR
                )
            ),
            "filesystem_after": after,
        },
        indent=2,
    )
)

print({
    "selected_model_files": len(
        after_models
    ),
    "selected_decrypted_gib": round(
        final_selected_bytes
        / GIB,
        3,
    ),
    "free_gib_after_models": round(
        after[
            "free_gib"
        ],
        3,
    ),
})


{
  "resolved_agent_config": "conf/common/agents/cicero.prototxt",
  "concrete_config_files": 8,
  "config_group_mounts": 0,
  "unresolved_includes": [],
  "selected_model_files": 67
}
MODEL blueprint.pt
MODEL cicero_imitation_bilateral_orders_prefix
MODEL cicero_imitation_bilateral_orders_prefix.dict
MODEL cicero_imitation_bilateral_orders_prefix.opt
MODEL dialogue
MODEL dialogue.dict
MODEL dialogue.opt
MODEL draw_classifier
MODEL draw_classifier.dict
MODEL draw_classifier.opt
MODEL imitation_intent
MODEL imitation_intent.dict
MODEL imitation_intent.opt
MODEL markus_5m_prompts.json
MODEL nonsense_ensemble/humanvsmodel_nonsense_classifier_denoising_cardinals
MODEL nonsense_ensemble/humanvsmodel_nonsense_classifier_denoising_cardinals.dict
MODEL nonsense_ensemble/humanvsmodel_nonsense_classifier_denoising_cardinals.opt
MODEL nonsense_ensemble/humanvsmodel_nonsense_classifier_denoising_justifications
MODEL nonsense_ensemble/humanvsmodel_nonsense_classifier_denoising_justifications.dict
M

Probe selected model sizes:   0%|          | 0/67 [00:00<?, ?file/s]

{
  "filesystem_before": {
    "total_bytes": 399327958007808,
    "used_bytes": 288593377820672,
    "free_bytes": 110734580187136,
    "free_gib": 103129.61431884766,
    "models_bytes": 33463639523,
    "models_encrypted_bytes": 0
  },
  "selected_file_count": 67,
  "missing_file_count": 0,
  "download_bytes": 0,
  "download_gib": 0.0,
  "estimated_final_selected_gib": 31.165442916564643,
  "largest_encrypted_file_gib": 0.0,
  "environment_reserve_gib": 8.0,
  "working_reserve_gib": 2.0,
  "required_free_gib": 10.0,
  "fits_current_filesystem": true
}


Fetch selected CICERO model subset:   0%|          | 0/67 [00:00<?, ?file/s]

{'selected_model_files': 67, 'selected_decrypted_gib': 31.165, 'free_gib_after_models': 103129.614}


## 6. Build CPython 3.7.17 and create an isolated virtual environment


In [7]:
import select


post_model_usage = shutil.disk_usage(
    PROJECT_ROOT
)

post_model_free_gib = float(
    post_model_usage.free
    / (
        1024
        ** 3
    )
)

minimum_runtime_free_gib = (
    ENVIRONMENT_RESERVE_GIB
    + WORKING_RESERVE_GIB
)

print({
    "free_gib_before_runtime_bootstrap": round(
        post_model_free_gib,
        3,
    ),
    "runtime_and_working_reserve_gib": minimum_runtime_free_gib,
})

if post_model_free_gib < minimum_runtime_free_gib:
    raise RuntimeError(
        "The selected weights fit, but insufficient free disk remains "
        "for the isolated runtime plus working reserve."
    )


bootstrap_records = []


def run_bootstrap_stage(
    stage,
    command,
    *,
    cwd=None,
    env=None,
):
    log_path = (
        ENGINE_DIR
        / f"{stage}.log"
    )

    command = [
        str(
            item
        )
        for item in command
    ]

    started = time.monotonic()
    last_display = started

    with log_path.open(
        "w",
        encoding="utf-8",
    ) as log_handle:
        process = subprocess.Popen(
            command,
            cwd=cwd,
            env=env,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
        )

        progress = tqdm(
            desc=stage,
            unit="log line",
            mininterval=2.0,
            miniters=100,
            dynamic_ncols=False,
        )

        assert process.stdout is not None

        while True:
            ready, _, _ = select.select(
                [
                    process.stdout
                ],
                [],
                [],
                5.0,
            )

            if ready:
                line = process.stdout.readline()

                if line:
                    log_handle.write(
                        line
                    )

                    log_handle.flush()

                    progress.update(
                        1
                    )

            elapsed = int(
                time.monotonic()
                - started
            )

            progress.set_postfix_str(
                f"running {elapsed // 60:02d}:{elapsed % 60:02d}",
                refresh=False,
            )

            now = time.monotonic()

            if (
                now
                - last_display
                >= 10.0
            ):
                progress.refresh()
                last_display = now

            returncode = process.poll()

            if returncode is not None:
                remainder = process.stdout.read()

                if remainder:
                    log_handle.write(
                        remainder
                    )

                    log_handle.flush()

                    progress.update(
                        len(
                            remainder.splitlines()
                        )
                    )

                break

        progress.close()

    record = {
        "stage": stage,
        "command": command,
        "returncode": int(
            returncode
        ),
        "elapsed_seconds": float(
            time.monotonic()
            - started
        ),
        "log": str(
            log_path
        ),
    }

    bootstrap_records.append(
        record
    )

    if returncode != 0:
        tail = "\n".join(
            log_path.read_text(
                errors="replace"
            ).splitlines()[
                -120:
            ]
        )

        raise RuntimeError(
            f"Stage {stage} failed; log: {log_path}\n\n"
            f"Last log lines:\n{tail}"
        )

    return record


def digest_file(
    path,
    algorithm,
):
    value = hashlib.new(
        algorithm
    )

    with Path(
        path
    ).open(
        "rb"
    ) as handle:
        for block in iter(
            lambda: handle.read(
                1024
                * 1024
            ),
            b"",
        ):
            value.update(
                block
            )

    return value.hexdigest()


RUNTIME_VENDOR = (
    PROJECT_ROOT
    / "vendor"
    / "cicero_runtime_v5"
)

SOURCE_CACHE = (
    RUNTIME_VENDOR
    / "sources"
)

BUILD_ROOT = (
    RUNTIME_VENDOR
    / "build"
)

OPENSSL_PREFIX = (
    RUNTIME_VENDOR
    / "openssl-1.1.1w"
)

PYTHON_BASE = (
    RUNTIME_VENDOR
    / "cpython-3.7.17"
)

LEGACY_ENV = (
    PROJECT_ROOT
    / ".venvs"
    / "cicero_full_agent_py37_source_v5"
)

for path in [
    SOURCE_CACHE,
    BUILD_ROOT,
    LEGACY_ENV.parent,
]:
    path.mkdir(
        parents=True,
        exist_ok=True,
    )


PYTHON_ARCHIVE = (
    SOURCE_CACHE
    / "Python-3.7.17.tar.xz"
)

OPENSSL_ARCHIVE = (
    SOURCE_CACHE
    / "openssl-1.1.1w.tar.gz"
)

PYTHON_SOURCE_URL = (
    "https://www.python.org/ftp/python/3.7.17/Python-3.7.17.tar.xz"
)

OPENSSL_SOURCE_URL = (
    "https://www.openssl.org/source/old/1.1.1/openssl-1.1.1w.tar.gz"
)

PYTHON_SOURCE_MD5 = (
    "dd94cab4541b57b88cf3dab32d6336e3"
)

OPENSSL_SOURCE_SHA256 = (
    "cf3098950cb4d853ad95c0841f1f9c6d3dc102dccfcacd521d93925208b76ac8"
)


def ensure_source(
    destination,
    url,
    algorithm,
    expected,
):
    destination = Path(
        destination
    )

    if (
        not destination.exists()
        or digest_file(
            destination,
            algorithm,
        )
        != expected
    ):
        destination.unlink(
            missing_ok=True
        )

        run_bootstrap_stage(
            f"download_{destination.name}",
            [
                "curl",
                "-L",
                "--fail",
                "--retry",
                "4",
                "--retry-delay",
                "3",
                "-o",
                destination,
                url,
            ],
        )

    actual = digest_file(
        destination,
        algorithm,
    )

    if actual != expected:
        raise RuntimeError(
            f"Digest mismatch for {destination.name}: "
            f"expected {expected}, got {actual}"
        )

    return {
        "path": str(
            destination
        ),
        "algorithm": algorithm,
        "digest": actual,
        "size_bytes": int(
            destination.stat().st_size
        ),
    }


python_source_record = ensure_source(
    PYTHON_ARCHIVE,
    PYTHON_SOURCE_URL,
    "md5",
    PYTHON_SOURCE_MD5,
)

openssl_source_record = ensure_source(
    OPENSSL_ARCHIVE,
    OPENSSL_SOURCE_URL,
    "sha256",
    OPENSSL_SOURCE_SHA256,
)


apt_packages = [
    "build-essential",
    "ca-certificates",
    "curl",
    "xz-utils",
    "perl",
    "pkg-config",
    "libffi-dev",
    "zlib1g-dev",
    "libbz2-dev",
    "libreadline-dev",
    "libsqlite3-dev",
    "libncursesw5-dev",
    "libgdbm-dev",
    "libgdbm-compat-dev",
    "liblzma-dev",
    "uuid-dev",
    "tk-dev",
]

apt_prefix = None

if shutil.which(
    "apt-get"
):
    if os.geteuid() == 0:
        apt_prefix = []
    elif shutil.which(
        "sudo"
    ):
        sudo_probe = subprocess.run(
            [
                "sudo",
                "-n",
                "true",
            ],
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            check=False,
        )

        if sudo_probe.returncode == 0:
            apt_prefix = [
                "sudo",
                "-n",
            ]

if apt_prefix is not None:
    run_bootstrap_stage(
        "apt_runtime_build_update",
        apt_prefix
        + [
            "apt-get",
            "update",
        ],
    )

    run_bootstrap_stage(
        "apt_runtime_build_dependencies",
        apt_prefix
        + [
            "apt-get",
            "install",
            "-y",
        ]
        + apt_packages,
    )
else:
    print(
        "No non-interactive host package installation is available; "
        "using the build headers already present."
    )


OPENSSL_BIN = (
    OPENSSL_PREFIX
    / "bin"
    / "openssl"
)

if not OPENSSL_BIN.exists():
    openssl_source_dir = (
        BUILD_ROOT
        / "openssl-1.1.1w"
    )

    if openssl_source_dir.exists():
        shutil.rmtree(
            openssl_source_dir
        )

    run_bootstrap_stage(
        "extract_openssl",
        [
            "tar",
            "--no-same-owner",
            "--no-same-permissions",
            "-xzf",
            OPENSSL_ARCHIVE,
            "-C",
            BUILD_ROOT,
        ],
    )

    run_bootstrap_stage(
        "configure_openssl",
        [
            "./config",
            f"--prefix={OPENSSL_PREFIX}",
            f"--openssldir={OPENSSL_PREFIX / 'ssl'}",
            "shared",
            "zlib",
        ],
        cwd=openssl_source_dir,
    )

    run_bootstrap_stage(
        "build_openssl",
        [
            "make",
            "-j2",
        ],
        cwd=openssl_source_dir,
    )

    run_bootstrap_stage(
        "install_openssl",
        [
            "make",
            "install_sw",
        ],
        cwd=openssl_source_dir,
    )

# Discover the actual private OpenSSL library directory rather than
# assuming the platform installed shared objects under "lib".
openssl_library_candidates = []

for pattern in [
    "libssl.so.1.1",
    "libcrypto.so.1.1",
]:
    openssl_library_candidates.extend(
        path.resolve()
        for path in OPENSSL_PREFIX.rglob(
            pattern
        )
        if path.is_file()
        or path.is_symlink()
    )

openssl_library_directories = sorted(
    {
        path.parent
        for path in openssl_library_candidates
    }
)

if not openssl_library_directories:
    raise RuntimeError(
        "OpenSSL installed, but libssl.so.1.1/libcrypto.so.1.1 "
        f"were not found under {OPENSSL_PREFIX}."
    )

# Prefer the directory containing both shared libraries.
OPENSSL_LIB_DIR = None

for candidate in openssl_library_directories:
    if (
        (
            candidate
            / "libssl.so.1.1"
        ).exists()
        and (
            candidate
            / "libcrypto.so.1.1"
        ).exists()
    ):
        OPENSSL_LIB_DIR = candidate
        break

if OPENSSL_LIB_DIR is None:
    OPENSSL_LIB_DIR = openssl_library_directories[
        0
    ]

openssl_runtime_env = os.environ.copy()

openssl_runtime_env[
    "LD_LIBRARY_PATH"
] = (
    str(
        OPENSSL_LIB_DIR
    )
    + os.pathsep
    + openssl_runtime_env.get(
        "LD_LIBRARY_PATH",
        "",
    )
)

openssl_probe = subprocess.run(
    [
        OPENSSL_BIN,
        "version",
        "-a",
    ],
    env=openssl_runtime_env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    check=False,
)

openssl_ldd = subprocess.run(
    [
        "ldd",
        str(
            OPENSSL_BIN
        ),
    ],
    env=openssl_runtime_env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    check=False,
)

(
    ENGINE_DIR
    / "openssl_probe.txt"
).write_text(
    (
        "OPENSSL_LIB_DIR="
        + str(
            OPENSSL_LIB_DIR
        )
        + "\n\n"
        + openssl_probe.stdout
        + "\n\nldd:\n"
        + openssl_ldd.stdout
    )
)

print(
    "OPENSSL_LIB_DIR:",
    OPENSSL_LIB_DIR,
)

print(
    openssl_probe.stdout
)

if (
    openssl_probe.returncode != 0
    or "OpenSSL 1.1.1w" not in openssl_probe.stdout
):
    raise RuntimeError(
        "Project-local OpenSSL 1.1.1w probe failed. "
        f"See {ENGINE_DIR / 'openssl_probe.txt'}."
    )


BASE_PYTHON = (
    PYTHON_BASE
    / "bin"
    / "python3.7"
)

if not BASE_PYTHON.exists():
    python_source_dir = (
        BUILD_ROOT
        / "Python-3.7.17"
    )

    if python_source_dir.exists():
        shutil.rmtree(
            python_source_dir
        )

    run_bootstrap_stage(
        "extract_python37",
        [
            "tar",
            "--no-same-owner",
            "--no-same-permissions",
            "-xJf",
            PYTHON_ARCHIVE,
            "-C",
            BUILD_ROOT,
        ],
    )

    python_build_env = os.environ.copy()

    python_build_env[
        "CPPFLAGS"
    ] = (
        f"-I{OPENSSL_PREFIX / 'include'}"
    )

    python_build_env[
        "LDFLAGS"
    ] = (
        f"-L{OPENSSL_LIB_DIR} "
        f"-Wl,-rpath,{OPENSSL_LIB_DIR} "
        f"-Wl,-rpath,{PYTHON_BASE / 'lib'}"
    )

    python_build_env[
        "PKG_CONFIG_PATH"
    ] = (
        str(
            OPENSSL_PREFIX
            / "lib"
            / "pkgconfig"
        )
        + os.pathsep
        + python_build_env.get(
            "PKG_CONFIG_PATH",
            "",
        )
    )

    python_build_env[
        "LD_LIBRARY_PATH"
    ] = (
        str(
            python_source_dir
        )
        + os.pathsep
        + str(
            OPENSSL_PREFIX
            / "lib"
        )
        + os.pathsep
        + python_build_env.get(
            "LD_LIBRARY_PATH",
            "",
        )
    )

    run_bootstrap_stage(
        "configure_python37",
        [
            "./configure",
            f"--prefix={PYTHON_BASE}",
            f"--with-openssl={OPENSSL_PREFIX}",
            "--enable-shared",
            "--with-ensurepip=install",
        ],
        cwd=python_source_dir,
        env=python_build_env,
    )

    run_bootstrap_stage(
        "build_python37",
        [
            "make",
            "-j2",
        ],
        cwd=python_source_dir,
        env=python_build_env,
    )

    run_bootstrap_stage(
        "install_python37",
        [
            "make",
            "install",
        ],
        cwd=python_source_dir,
        env=python_build_env,
    )


base_runtime_env = os.environ.copy()

base_runtime_env[
    "LD_LIBRARY_PATH"
] = (
    str(
        PYTHON_BASE
        / "lib"
    )
    + os.pathsep
    + str(
        OPENSSL_LIB_DIR
    )
    + os.pathsep
    + base_runtime_env.get(
        "LD_LIBRARY_PATH",
        "",
    )
)

base_probe = subprocess.run(
    [
        BASE_PYTHON,
        "-c",
        (
            "import bz2,ctypes,json,lzma,sqlite3,ssl,sys,zlib; "
            "print(json.dumps({"
            "'executable':sys.executable,"
            "'version':sys.version,"
            "'openssl':ssl.OPENSSL_VERSION,"
            "'zlib':zlib.ZLIB_VERSION,"
            "'sqlite':sqlite3.sqlite_version,"
            "'ctypes':True,"
            "'bz2':True,"
            "'lzma':True"
            "}))"
        ),
    ],
    env=base_runtime_env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    check=False,
)

(
    ENGINE_DIR
    / "source_python_probe.txt"
).write_text(
    base_probe.stdout
)

print(
    base_probe.stdout
)

if (
    base_probe.returncode != 0
    or "3.7.17" not in base_probe.stdout
    or "OpenSSL 1.1.1w" not in base_probe.stdout
):
    raise RuntimeError(
        "Source-built Python 3.7.17 validation failed; "
        "see source_python_probe.txt."
    )


recreate_venv = bool(
    int(
        os.environ.get(
            "LR_14_RECREATE_VENV",
            "0",
        )
    )
)

if (
    recreate_venv
    and LEGACY_ENV.exists()
):
    shutil.rmtree(
        LEGACY_ENV
    )

LEGACY_PYTHON = (
    LEGACY_ENV
    / "bin"
    / "python"
)

if not LEGACY_PYTHON.exists():
    run_bootstrap_stage(
        "create_python37_venv",
        [
            BASE_PYTHON,
            "-m",
            "venv",
            "--copies",
            LEGACY_ENV,
        ],
        env=base_runtime_env,
    )


venv_include_root = (
    LEGACY_ENV
    / "include"
)

venv_include_root.mkdir(
    parents=True,
    exist_ok=True,
)

base_include = (
    PYTHON_BASE
    / "include"
    / "python3.7m"
)

venv_include = (
    venv_include_root
    / "python3.7m"
)

if (
    base_include.exists()
    and not venv_include.exists()
):
    venv_include.symlink_to(
        base_include,
        target_is_directory=True,
    )

venv_lib = (
    LEGACY_ENV
    / "lib"
)

venv_lib.mkdir(
    parents=True,
    exist_ok=True,
)

for library_name in [
    "libpython3.7m.so",
    "libpython3.7m.so.1.0",
]:
    source_library = (
        PYTHON_BASE
        / "lib"
        / library_name
    )

    target_library = (
        venv_lib
        / library_name
    )

    if (
        source_library.exists()
        and not target_library.exists()
    ):
        target_library.symlink_to(
            source_library
        )


legacy_runtime_env = base_runtime_env.copy()

legacy_runtime_env[
    "PATH"
] = (
    str(
        LEGACY_ENV
        / "bin"
    )
    + os.pathsep
    + os.environ.get(
        "PATH",
        "",
    )
)

legacy_runtime_env[
    "VIRTUAL_ENV"
] = str(
    LEGACY_ENV
)

legacy_runtime_env[
    "PYTHONNOUSERSITE"
] = "1"

legacy_runtime_env[
    "LD_LIBRARY_PATH"
] = (
    str(
        LEGACY_ENV
        / "lib"
    )
    + os.pathsep
    + str(
        PYTHON_BASE
        / "lib"
    )
    + os.pathsep
    + str(
        OPENSSL_LIB_DIR
    )
    + os.pathsep
    + os.environ.get(
        "LD_LIBRARY_PATH",
        "",
    )
)

python_probe = subprocess.run(
    [
        LEGACY_PYTHON,
        "-c",
        (
            "import json,ssl,sys; "
            "print(json.dumps({"
            "'executable':sys.executable,"
            "'version':sys.version,"
            "'prefix':sys.prefix,"
            "'base_prefix':sys.base_prefix,"
            "'openssl':ssl.OPENSSL_VERSION"
            "}))"
        ),
    ],
    env=legacy_runtime_env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    check=False,
)

if python_probe.returncode != 0:
    raise RuntimeError(
        "Python 3.7 virtual-environment probe failed."
    )

source_runtime_bootstrap = {
    "python_source": python_source_record,
    "openssl_source": openssl_source_record,
    "openssl_prefix": str(
        OPENSSL_PREFIX
    ),
    "openssl_library_directory": str(
        OPENSSL_LIB_DIR
    ),
    "python_base_prefix": str(
        PYTHON_BASE
    ),
    "virtual_environment": str(
        LEGACY_ENV
    ),
    "python_probe": python_probe.stdout.strip(),
    "package_installation_mode": "pip",
}

(
    ENGINE_DIR
    / "source_runtime_bootstrap.json"
).write_text(
    json.dumps(
        source_runtime_bootstrap,
        indent=2,
    )
)

(
    ENGINE_DIR
    / "bootstrap_stages.json"
).write_text(
    json.dumps(
        bootstrap_records,
        indent=2,
    )
)

print(
    json.dumps(
        source_runtime_bootstrap,
        indent=2,
    )
)


{'free_gib_before_runtime_bootstrap': 103129.614, 'runtime_and_working_reserve_gib': 10.0}


apt_runtime_build_update: 0log line [00:00, ?log line/s]

apt_runtime_build_dependencies: 0log line [00:00, ?log line/s]

OPENSSL_LIB_DIR: /workspace/latent-reservations/vendor/cicero_runtime_v5/openssl-1.1.1w/lib
OpenSSL 1.1.1w  11 Sep 2023
built on: Sat Aug 15 10:28:44 2026 UTC
platform: linux-x86_64
options:  bn(64,64) rc4(8x,int) des(int) idea(int) blowfish(ptr) 
compiler: gcc -fPIC -pthread -m64 -Wa,--noexecstack -Wall -O3 -DOPENSSL_USE_NODELETE -DL_ENDIAN -DOPENSSL_PIC -DOPENSSL_CPUID_OBJ -DOPENSSL_IA32_SSE2 -DOPENSSL_BN_ASM_MONT -DOPENSSL_BN_ASM_MONT5 -DOPENSSL_BN_ASM_GF2m -DSHA1_ASM -DSHA256_ASM -DSHA512_ASM -DKECCAK1600_ASM -DRC4_ASM -DMD5_ASM -DAESNI_ASM -DVPAES_ASM -DGHASH_ASM -DECP_NISTZ256_ASM -DX25519_ASM -DPOLY1305_ASM -DZLIB -DNDEBUG
OPENSSLDIR: "/workspace/latent-reservations/vendor/cicero_runtime_v5/openssl-1.1.1w/ssl"
ENGINESDIR: "/workspace/latent-reservations/vendor/cicero_runtime_v5/openssl-1.1.1w/lib/engines-1.1"
Seeding source: os-specific

{"executable": "/workspace/latent-reservations/vendor/cicero_runtime_v5/cpython-3.7.17/bin/python3.7", "version": "3.7.17 (default, Aug 15 2026

## 7. Install the released Python stack with `pip`


In [8]:
import select

environment_records = []

PIP = (
    LEGACY_ENV
    / "bin"
    / "pip"
)

if not PIP.exists():
    raise FileNotFoundError(
        PIP
    )


def run_logged_stage(
    stage,
    command,
    *,
    cwd=None,
    env=None,
):
    log_path = (
        ENGINE_DIR
        / f"{stage}.log"
    )

    command = [
        str(
            item
        )
        for item in command
    ]

    started = time.monotonic()
    last_display = started

    with log_path.open(
        "w",
        encoding="utf-8",
    ) as log_handle:
        process = subprocess.Popen(
            command,
            cwd=cwd,
            env=env,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
        )

        progress = tqdm(
            desc=stage,
            unit="log line",
            mininterval=2.0,
            miniters=100,
            dynamic_ncols=False,
        )

        assert process.stdout is not None

        while True:
            ready, _, _ = select.select(
                [
                    process.stdout
                ],
                [],
                [],
                5.0,
            )

            if ready:
                line = process.stdout.readline()

                if line:
                    log_handle.write(
                        line
                    )
                    log_handle.flush()

                    progress.update(
                        1
                    )

            elapsed = int(
                time.monotonic()
                - started
            )

            progress.set_postfix_str(
                f"running {elapsed // 60:02d}:{elapsed % 60:02d}",
                refresh=False,
            )

            now = time.monotonic()

            if (
                now
                - last_display
                >= 10.0
            ):
                progress.refresh()
                last_display = now

            returncode = process.poll()

            if returncode is not None:
                remainder = process.stdout.read()

                if remainder:
                    log_handle.write(
                        remainder
                    )
                    log_handle.flush()

                    progress.update(
                        len(
                            remainder.splitlines()
                        )
                    )

                break

        progress.close()

    record = {
        "stage": stage,
        "command": command,
        "returncode": int(
            returncode
        ),
        "elapsed_seconds": float(
            time.monotonic()
            - started
        ),
        "log": str(
            log_path
        ),
    }

    environment_records.append(
        record
    )

    if returncode != 0:
        tail = "\n".join(
            log_path.read_text(
                errors="replace"
            ).splitlines()[
                -120:
            ]
        )

        raise RuntimeError(
            f"Stage {stage} failed; log: {log_path}\n\n"
            f"Last log lines:\n{tail}"
        )

    return record


run_logged_stage(
    "pip_bootstrap_tools",
    [
        LEGACY_PYTHON,
        "-m",
        "pip",
        "install",
        "--no-cache-dir",
        "pip==23.3.2",
        "setuptools==65.6.3",
        "wheel==0.41.3",
    ],
)

run_logged_stage(
    "pip_torch_cuda110",
    [
        LEGACY_PYTHON,
        "-m",
        "pip",
        "install",
        "--no-cache-dir",
        "torch==1.7.1+cu110",
        "torchvision==0.8.2+cu110",
        "torchaudio==0.7.2",
        "-f",
        "https://download.pytorch.org/whl/torch_stable.html",
    ],
)

source_requirements = (
    CICERO_REPO
    / "requirements.txt"
)

pip_requirements = (
    ENGINE_DIR
    / "requirements_pip_v5.txt"
)

requirement_lines = []

for line in source_requirements.read_text().splitlines():
    stripped = line.strip()

    # The released requirements intentionally leave torch unpinned.
    # The CUDA 11.0 wheel above is the authoritative torch installation.
    if re.match(
        r"^torch(?:\s*(?:#.*)?)?$",
        stripped,
    ):
        continue

    requirement_lines.append(
        line
    )

pip_requirements.write_text(
    "\n".join(
        requirement_lines
    )
    + "\n"
)

run_logged_stage(
    "pip_requirements",
    [
        LEGACY_PYTHON,
        "-m",
        "pip",
        "install",
        "--no-cache-dir",
        "-r",
        pip_requirements,
    ],
    cwd=CICERO_REPO,
)

run_logged_stage(
    "pip_build_dependencies",
    [
        LEGACY_PYTHON,
        "-m",
        "pip",
        "install",
        "--no-cache-dir",
        "cmake==3.22.6",
        "pybind11==2.10.4",
        "protobuf==3.19.1",
        "mypy-protobuf==2.10",
    ],
)

# The protobuf Python package does not provide the `protoc` executable used by
# the repository Makefile. Install the released compiler version expected by
# this codebase into the project-local runtime and put it on PATH.
PROTOC_VERSION = "3.19.1"

PROTOC_ROOT = (
    RUNTIME_VENDOR
    / f"protoc-{PROTOC_VERSION}"
)

PROTOC_BIN_DIR = (
    PROTOC_ROOT
    / "bin"
)

PROTOC_EXECUTABLE = (
    PROTOC_BIN_DIR
    / "protoc"
)

PROTOC_ARCHIVE = (
    RUNTIME_VENDOR
    / f"protoc-{PROTOC_VERSION}-linux-x86_64.zip"
)

PROTOC_URL = (
    "https://github.com/protocolbuffers/protobuf/releases/download/"
    f"v{PROTOC_VERSION}/"
    f"protoc-{PROTOC_VERSION}-linux-x86_64.zip"
)

if not PROTOC_EXECUTABLE.exists():
    if not PROTOC_ARCHIVE.exists():
        protoc_download = capture_command(
            [
                "curl",
                "-L",
                "--fail",
                "--retry",
                "3",
                "-o",
                PROTOC_ARCHIVE,
                PROTOC_URL,
            ]
        )

        if protoc_download[
            "returncode"
        ] != 0:
            raise RuntimeError(
                "Could not download the project-local protoc 3.19.1 release."
            )

    protoc_staging = (
        RUNTIME_VENDOR
        / "_protoc_3_19_1_extract"
    )

    if protoc_staging.exists():
        shutil.rmtree(
            protoc_staging
        )

    protoc_staging.mkdir(
        parents=True,
        exist_ok=True,
    )

    shutil.unpack_archive(
        str(
            PROTOC_ARCHIVE
        ),
        str(
            protoc_staging
        ),
        "zip",
    )

    if PROTOC_ROOT.exists():
        shutil.rmtree(
            PROTOC_ROOT
        )

    shutil.move(
        str(
            protoc_staging
        ),
        str(
            PROTOC_ROOT
        ),
    )

if not PROTOC_EXECUTABLE.exists():
    raise FileNotFoundError(
        PROTOC_EXECUTABLE
    )

PROTOC_EXECUTABLE.chmod(
    PROTOC_EXECUTABLE.stat().st_mode
    | 0o111
)

protoc_version_probe = capture_command(
    [
        PROTOC_EXECUTABLE,
        "--version",
    ]
)

print(
    protoc_version_probe[
        "output"
    ]
)

if (
    protoc_version_probe[
        "returncode"
    ] != 0
    or protoc_version_probe[
        "output"
    ].strip()
    != "libprotoc 3.19.1"
):
    raise RuntimeError(
        "Project-local protoc version is not exactly 3.19.1."
    )

PROTOC_GEN_MYPY = (
    LEGACY_ENV
    / "bin"
    / "protoc-gen-mypy"
)

if not PROTOC_GEN_MYPY.exists():
    raise FileNotFoundError(
        "mypy-protobuf installed without the protoc-gen-mypy executable: "
        f"{PROTOC_GEN_MYPY}"
    )

# CICERO's released installation requires Go for the bundled gRPC/BoringSSL path.
# Reuse a host Go installation when present; otherwise install a project-local
# official Go binary release without modifying system directories.
HOST_GO = shutil.which(
    "go"
)

GO_ROOT = (
    PROJECT_ROOT
    / "vendor"
    / "go1.17.13"
)

GO_BIN_DIR = (
    GO_ROOT
    / "bin"
)

if HOST_GO is None:
    GO_ARCHIVE = (
        PROJECT_ROOT
        / "vendor"
        / "go1.17.13.linux-amd64.tar.gz"
    )

    GO_ARCHIVE.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    if not GO_ARCHIVE.exists():
        go_download = capture_command(
            [
                "curl",
                "-L",
                "--fail",
                "-o",
                GO_ARCHIVE,
                "https://go.dev/dl/go1.17.13.linux-amd64.tar.gz",
            ]
        )

        if go_download[
            "returncode"
        ] != 0:
            raise RuntimeError(
                "Could not download the project-local Go runtime."
            )

    if not (
        GO_BIN_DIR
        / "go"
    ).exists():
        staging = (
            PROJECT_ROOT
            / "vendor"
            / "_go117_extract"
        )

        if staging.exists():
            shutil.rmtree(
                staging
            )

        staging.mkdir(
            parents=True,
            exist_ok=True,
        )

        extract = capture_command(
            [
                "tar",
                "-xzf",
                GO_ARCHIVE,
                "-C",
                staging,
            ]
        )

        if extract[
            "returncode"
        ] != 0:
            raise RuntimeError(
                "Could not extract the project-local Go runtime."
            )

        extracted_go = (
            staging
            / "go"
        )

        if GO_ROOT.exists():
            shutil.rmtree(
                GO_ROOT
            )

        shutil.move(
            str(
                extracted_go
            ),
            str(
                GO_ROOT
            ),
        )

        shutil.rmtree(
            staging
        )

    GO_EXECUTABLE = (
        GO_BIN_DIR
        / "go"
    )
else:
    GO_EXECUTABLE = Path(
        HOST_GO
    ).resolve()

    GO_BIN_DIR = (
        GO_EXECUTABLE.parent
    )

legacy_build_env = os.environ.copy()

# Do not inherit notebook-host Python package routing into the isolated
# Python 3.7 build.
for key in [
    "PYTHONPATH",
    "PYTHONHOME",
]:
    legacy_build_env.pop(
        key,
        None,
    )

for key in [
    "CC",
    "CXX",
    "CPP",
    "CFLAGS",
    "CPPFLAGS",
    "LDFLAGS",
    "CMAKE_TOOLCHAIN_FILE",
    "CMAKE_C_COMPILER",
    "CMAKE_CXX_COMPILER",
]:
    legacy_build_env.pop(
        key,
        None,
    )

legacy_build_env[
    "PATH"
] = (
    str(
        PROTOC_BIN_DIR
    )
    + os.pathsep
    + str(
        LEGACY_ENV
        / "bin"
    )
    + os.pathsep
    + str(
        GO_BIN_DIR
    )
    + os.pathsep
    + os.environ.get(
        "PATH",
        "",
    )
)

legacy_build_env[
    "PYTHON"
] = str(
    LEGACY_PYTHON
)

legacy_build_env[
    "PYTHON_EXECUTABLE"
] = str(
    LEGACY_PYTHON
)

legacy_build_env[
    "VIRTUAL_ENV"
] = str(
    LEGACY_ENV
)

pybind11_routing_probe = capture_command(
    [
        LEGACY_PYTHON,
        "-c",
        (
            "import json,pathlib,pybind11; "
            "cmake_dir=pathlib.Path(pybind11.get_cmake_dir()).resolve(); "
            "print(json.dumps({"
            "'version':pybind11.__version__,"
            "'cmake_dir':str(cmake_dir),"
            "'prefix':str(cmake_dir.parents[2])"
            "}))"
        ),
    ],
    env=legacy_build_env,
)

if pybind11_routing_probe[
    "returncode"
] != 0:
    raise RuntimeError(
        "Could not resolve pybind11's CMake package from Python 3.7."
    )

pybind11_build_routing = json.loads(
    pybind11_routing_probe[
        "output"
    ].strip().splitlines()[
        -1
    ]
)

if pybind11_build_routing[
    "version"
] != "2.10.4":
    raise RuntimeError(
        "Unexpected pybind11 version in the isolated build: "
        + json.dumps(
            pybind11_build_routing,
            sort_keys=True,
        )
    )

PYBIND11_CMAKE_DIR = Path(
    pybind11_build_routing[
        "cmake_dir"
    ]
)

PYBIND11_CMAKE_PREFIX = Path(
    pybind11_build_routing[
        "prefix"
    ]
)

if not (
    PYBIND11_CMAKE_DIR
    / "pybind11Config.cmake"
).exists():
    raise FileNotFoundError(
        PYBIND11_CMAKE_DIR
        / "pybind11Config.cmake"
    )

# Resolve Torch from the isolated interpreter itself.  The project has had
# earlier CMake builds under the notebook host Python, so relying on generic
# prefix discovery can mix host headers with the intended 1.7.1 libraries.
torch_build_routing_probe = capture_command(
    [
        LEGACY_PYTHON,
        "-c",
        (
            "import json,pathlib,torch; "
            "root=pathlib.Path(torch.__file__).resolve().parent; "
            "print(json.dumps({"
            "'version':torch.__version__,"
            "'cuda':torch.version.cuda,"
            "'root':str(root),"
            "'cmake_prefix':str(root/'share'/'cmake'),"
            "'torch_dir':str(root/'share'/'cmake'/'Torch'),"
            "'lib_dir':str(root/'lib'),"
            "'include_dir':str(root/'include'),"
            "'cxx11_abi':bool(torch._C._GLIBCXX_USE_CXX11_ABI)"
            "}))"
        ),
    ],
    env=legacy_build_env,
)

if torch_build_routing_probe[
    "returncode"
] != 0:
    raise RuntimeError(
        "Could not resolve the isolated PyTorch CMake routing."
    )

torch_build_routing = json.loads(
    torch_build_routing_probe[
        "output"
    ].strip().splitlines()[
        -1
    ]
)

if not torch_build_routing[
    "version"
].startswith(
    "1.7.1"
):
    raise RuntimeError(
        "The isolated build is not resolving PyTorch 1.7.1: "
        + json.dumps(
            torch_build_routing,
            sort_keys=True,
        )
    )

TORCH_PACKAGE_ROOT = Path(
    torch_build_routing[
        "root"
    ]
)

TORCH_CMAKE_PREFIX = Path(
    torch_build_routing[
        "cmake_prefix"
    ]
)

TORCH_CMAKE_DIR = Path(
    torch_build_routing[
        "torch_dir"
    ]
)

TORCH_LIB_DIR = Path(
    torch_build_routing[
        "lib_dir"
    ]
)

for required_path in [
    TORCH_CMAKE_DIR
    / "TorchConfig.cmake",
    TORCH_LIB_DIR,
]:
    if not required_path.exists():
        raise FileNotFoundError(
            required_path
        )

legacy_build_env[
    "CMAKE_PREFIX_PATH"
] = os.pathsep.join(
    [
        str(
            PYBIND11_CMAKE_PREFIX
        ),
        str(
            PYBIND11_CMAKE_DIR
        ),
        str(
            TORCH_CMAKE_PREFIX
        ),
        str(
            LEGACY_ENV
        ),
        str(
            PYTHON_BASE
        ),
    ]
)

legacy_build_env[
    "PYBIND11_CMAKE_DIR"
] = str(
    PYBIND11_CMAKE_DIR
)

legacy_build_env[
    "CMAKE_LIBRARY_PATH"
] = os.pathsep.join(
    [
        str(
            TORCH_LIB_DIR
        ),
        str(
            LEGACY_ENV
            / "lib"
        ),
        str(
            PYTHON_BASE
            / "lib"
        ),
    ]
)

legacy_build_env[
    "LD_LIBRARY_PATH"
] = os.pathsep.join(
    [
        str(
            TORCH_LIB_DIR
        ),
        legacy_runtime_env[
            "LD_LIBRARY_PATH"
        ],
    ]
)

legacy_build_env[
    "PYTHONNOUSERSITE"
] = "1"

# Modern compiler/header stacks no longer provide a few standard-library
# declarations transitively that this older source assumes are already present.
# Supply only the standard headers demonstrated by the build logs, without
# modifying the frozen repository checkout.
POSTMAN_COMPAT_HEADER = (
    ENGINE_DIR
    / "postman_compat.hpp"
)

POSTMAN_COMPAT_HEADER.write_text(
    "#pragma once\n"
    "#include <cstdint>\n"
    "#include <stdexcept>\n"
)

existing_cxxflags = legacy_build_env.get(
    "CXXFLAGS",
    "",
).strip()

legacy_build_env[
    "CXXFLAGS"
] = (
    existing_cxxflags
    + f" -include {POSTMAN_COMPAT_HEADER}"
).strip()

nvcc_path = shutil.which(
    "nvcc"
)

if nvcc_path:
    cuda_home = Path(
        nvcc_path
    ).resolve().parent.parent

    legacy_build_env[
        "CUDA_HOME"
    ] = str(
        cuda_home
    )

tool_probe = capture_command(
    [
        "bash",
        "-c",
        (
            "set -e; "
            "echo python=$(command -v python); "
            "python --version; "
            "echo pip=$(command -v pip); "
            "echo cmake=$(command -v cmake); "
            "cmake --version | head -1; "
            "echo gcc=$(command -v gcc); "
            "gcc --version | head -1; "
            "echo gxx=$(command -v g++); "
            "g++ --version | head -1; "
            "echo go=$(command -v go); "
            "go version; "
            "echo protoc=$(command -v protoc); "
            "protoc --version; "
            "echo protoc_gen_mypy=$(command -v protoc-gen-mypy); "
            "echo pybind11_cmake_dir=$PYBIND11_CMAKE_DIR; "
            "echo nvcc=$(command -v nvcc || true); "
            "echo CMAKE_PREFIX_PATH=$CMAKE_PREFIX_PATH; "
            "echo CMAKE_LIBRARY_PATH=$CMAKE_LIBRARY_PATH; "
            "echo LD_LIBRARY_PATH=$LD_LIBRARY_PATH; "
            "echo CXXFLAGS=$CXXFLAGS"
        ),
    ],
    cwd=CICERO_REPO,
    env=legacy_build_env,
)

(
    ENGINE_DIR
    / "pip_runtime_tool_probe.txt"
).write_text(
    tool_probe[
        "output"
    ]
)

print(
    tool_probe[
        "output"
    ]
)

if tool_probe[
    "returncode"
] != 0:
    raise RuntimeError(
        "Runtime/build-tool routing probe failed."
    )

torch_probe = capture_command(
    [
        LEGACY_PYTHON,
        "-c",
        (
            "import json,torch; "
            "print(json.dumps({"
            "'torch':torch.__version__,"
            "'cuda_build':torch.version.cuda,"
            "'cuda_available':torch.cuda.is_available()"
            "}))"
        ),
    ],
    env=legacy_build_env,
)

(
    ENGINE_DIR
    / "pip_torch_probe.txt"
).write_text(
    torch_probe[
        "output"
    ]
)

print(
    torch_probe[
        "output"
    ]
)

if torch_probe[
    "returncode"
] != 0:
    raise RuntimeError(
        "PyTorch 1.7.1 runtime probe failed."
    )

compatibility_payload = {
    "python_environment": str(
        LEGACY_ENV
    ),
    "python_base": str(
        PYTHON_BASE
    ),
    "protoc": {
        "version": PROTOC_VERSION,
        "executable": str(
            PROTOC_EXECUTABLE
        ),
        "mypy_plugin": str(
            PROTOC_GEN_MYPY
        ),
    },
    "pybind11_build_routing": pybind11_build_routing,
    "pybind11_cmake_prefix": str(
        PYBIND11_CMAKE_PREFIX
    ),
    "pybind11_cmake_dir": str(
        PYBIND11_CMAKE_DIR
    ),
    "torch_build_routing": torch_build_routing,
    "torch_cmake_prefix": str(
        TORCH_CMAKE_PREFIX
    ),
    "torch_cmake_dir": str(
        TORCH_CMAKE_DIR
    ),
    "torch_library_directory": str(
        TORCH_LIB_DIR
    ),
    "openssl_prefix": str(
        OPENSSL_PREFIX
    ),
    "openssl_library_directory": str(
        OPENSSL_LIB_DIR
    ),
    "package_installer": "pip",
    "cmake": str(
        LEGACY_ENV
        / "bin"
        / "cmake"
    ),
    "go": str(
        GO_EXECUTABLE
    ),
    "host_gcc": shutil.which(
        "gcc"
    ),
    "host_gxx": shutil.which(
        "g++"
    ),
    "cxxflags": legacy_build_env[
        "CXXFLAGS"
    ],
    "postman_compat_header": str(
        POSTMAN_COMPAT_HEADER
    ),
    "nvcc": nvcc_path,
    "cuda_home": legacy_build_env.get(
        "CUDA_HOME"
    ),
}

(
    ENGINE_DIR
    / "compatibility_flags.json"
).write_text(
    json.dumps(
        compatibility_payload,
        indent=2,
    )
)


pip_bootstrap_tools: 0log line [00:00, ?log line/s]

pip_torch_cuda110: 0log line [00:00, ?log line/s]

pip_requirements: 0log line [00:00, ?log line/s]

pip_build_dependencies: 0log line [00:00, ?log line/s]

libprotoc 3.19.1

python=/workspace/latent-reservations/.venvs/cicero_full_agent_py37_source_v5/bin/python
Python 3.7.17
pip=/workspace/latent-reservations/.venvs/cicero_full_agent_py37_source_v5/bin/pip
cmake=/workspace/latent-reservations/.venvs/cicero_full_agent_py37_source_v5/bin/cmake
cmake version 3.22.6
gcc=/usr/bin/gcc
gcc (Ubuntu 13.3.0-6ubuntu2~24.04) 13.3.0
gxx=/usr/bin/g++
g++ (Ubuntu 13.3.0-6ubuntu2~24.04) 13.3.0
go=/workspace/latent-reservations/vendor/go1.17.13/bin/go
go version go1.17.13 linux/amd64
protoc=/workspace/latent-reservations/vendor/cicero_runtime_v5/protoc-3.19.1/bin/protoc
libprotoc 3.19.1
protoc_gen_mypy=/workspace/latent-reservations/.venvs/cicero_full_agent_py37_source_v5/bin/protoc-gen-mypy
pybind11_cmake_dir=/workspace/latent-reservations/.venvs/cicero_full_agent_py37_source_v5/lib/python3.7/site-packages/pybind11/share/cmake/pybind11
nvcc=/usr/local/cuda/bin/nvcc
CMAKE_PREFIX_PATH=/workspace/latent-reservations/.venvs/cicero_full_agent_py37_source_v5/

3163

## 8. Build Postman and the released CICERO stack


In [9]:
STACK_MARKER = (
    LEGACY_ENV
    / ".latent_reservations_cicero_stack_v5_7"
)

POSTMAN_ROOT = (
    CICERO_REPO
    / "thirdparty"
    / "github"
    / "fairinternal"
    / "postman"
)

POSTMAN_NEST_DIR = (
    POSTMAN_ROOT
    / "nest"
)

POSTMAN_DIR = (
    POSTMAN_ROOT
    / "postman"
)

POSTMAN_CMAKE_BUILD = (
    POSTMAN_ROOT
    / "cmake_build"
    / "postman"
)

if not STACK_MARKER.exists():
    # A completed Postman build is reusable across notebook revisions. Probe it
    # first so a later dipcc failure does not force the large gRPC build again.
    postman_import_probe = capture_command(
        [
            LEGACY_PYTHON,
            "-c",
            (
                "import sys; "
                "import postman; "
                "print('python', sys.executable); "
                "print('postman', postman.__file__)"
            ),
        ],
        cwd=CICERO_REPO,
        env=legacy_build_env,
    )

    if postman_import_probe[
        "returncode"
    ] != 0:
        run_logged_stage(
            "pip_postman_nest",
            [
                LEGACY_PYTHON,
                "-m",
                "pip",
                "install",
                "--no-cache-dir",
                "-e",
                POSTMAN_NEST_DIR,
            ],
            cwd=CICERO_REPO,
            env=legacy_build_env,
        )

        if POSTMAN_CMAKE_BUILD.exists():
            shutil.rmtree(
                POSTMAN_CMAKE_BUILD
            )

        print(
            "Cleared Postman CMake cache:",
            POSTMAN_CMAKE_BUILD,
        )

        # The old source assumes some standard-library declarations arrive
        # transitively. The build environment force-includes a tiny compatibility
        # header containing only the standard headers demonstrated missing by logs.
        run_logged_stage(
            "pip_postman",
            [
                LEGACY_PYTHON,
                "-m",
                "pip",
                "install",
                "--no-cache-dir",
                "-e",
                POSTMAN_DIR,
            ],
            cwd=CICERO_REPO,
            env=legacy_build_env,
        )

        postman_log = (
            ENGINE_DIR
            / "pip_postman.log"
        ).read_text(
            errors="replace"
        )

        if "/usr/local/bin/python" in postman_log:
            raise RuntimeError(
                "Postman resolved the notebook host Python instead of "
                "the isolated Python 3.7 interpreter."
            )

        postman_import_probe = capture_command(
            [
                LEGACY_PYTHON,
                "-c",
                (
                    "import sys; "
                    "import postman; "
                    "print('python', sys.executable); "
                    "print('postman', postman.__file__)"
                ),
            ],
            cwd=CICERO_REPO,
            env=legacy_build_env,
        )
    else:
        print(
            "Reusing existing Postman installation."
        )

    (
        RUNTIME_DIR
        / "postman_import_probe.txt"
    ).write_text(
        postman_import_probe[
            "output"
        ]
    )

    print(
        postman_import_probe[
            "output"
        ]
    )

    if postman_import_probe[
        "returncode"
    ] != 0:
        raise RuntimeError(
            "Postman could not be imported in Python 3.7."
        )

    DIPCC_BUILD_DIR = (
        CICERO_REPO
        / "dipcc"
        / "build"
    )

    PYDIPCC_GLOB = (
        CICERO_REPO
        / "fairdiplomacy"
    )

    def probe_pydipcc():
        return capture_command(
            [
                LEGACY_PYTHON,
                "-c",
                (
                    "import json,sys,torch; "
                    "from fairdiplomacy import pydipcc; "
                    "print(json.dumps({"
                    "'python':sys.executable,"
                    "'torch':torch.__version__,"
                    "'pydipcc':pydipcc.__file__"
                    "}))"
                ),
            ],
            cwd=CICERO_REPO,
            env=legacy_build_env,
        )

    existing_pydipcc_probe = probe_pydipcc()

    reuse_pydipcc = (
        existing_pydipcc_probe[
            "returncode"
        ]
        == 0
        and '"torch": "1.7.1'
        in existing_pydipcc_probe[
            "output"
        ]
        and "cpython-37m"
        in existing_pydipcc_probe[
            "output"
        ]
    )

    if reuse_pydipcc:
        print(
            "Reusing existing Python 3.7 pydipcc build."
        )

        print(
            existing_pydipcc_probe[
                "output"
            ]
        )

    else:
        # Remove only generated dipcc state and any stale extension before
        # configuring against the isolated Python 3.7 / PyTorch 1.7.1 stack.
        if DIPCC_BUILD_DIR.exists():
            shutil.rmtree(
                DIPCC_BUILD_DIR
            )

        for shared_object in PYDIPCC_GLOB.glob(
            "pydipcc*.so"
        ):
            shared_object.unlink()

        print(
            "Cleared generated dipcc state:",
            DIPCC_BUILD_DIR,
        )

    run_logged_stage(
        "pip_repo",
        [
            LEGACY_PYTHON,
            "-m",
            "pip",
            "install",
            "--no-cache-dir",
            "-e",
            CICERO_REPO,
            "-vv",
        ],
        cwd=CICERO_REPO,
        env=legacy_build_env,
    )

    build_dependency_probe = capture_command(
        [
            "bash",
            "-c",
            (
                "set -e; "
                "echo protoc=$(command -v protoc); "
                "protoc --version; "
                "echo protoc-gen-mypy=$(command -v protoc-gen-mypy); "
                "test -x $(command -v protoc-gen-mypy); "
                "python -c \"import pybind11; "
                "print('pybind11', pybind11.__version__); "
                "print('cmake', pybind11.get_cmake_dir())\""
            ),
        ],
        cwd=CICERO_REPO,
        env=legacy_build_env,
    )

    (
        ENGINE_DIR
        / "build_dependency_probe.txt"
    ).write_text(
        build_dependency_probe[
            "output"
        ]
    )

    print(
        build_dependency_probe[
            "output"
        ]
    )

    if (
        build_dependency_probe[
            "returncode"
        ]
        != 0
        or "libprotoc 3.19.1"
        not in build_dependency_probe[
            "output"
        ]
        or "pybind11 2.10.4"
        not in build_dependency_probe[
            "output"
        ]
    ):
        raise RuntimeError(
            "CICERO native build dependencies are not routed to the "
            "isolated Python 3.7 environment."
        )

    # Root `make` also builds the old self-play C++ test/training branch.
    # Notebook 14 needs generated config protobufs plus the Python game-engine
    # extension, so build those branches independently.
    run_logged_stage(
        "legacy_make_protos",
        [
            "make",
            "protos",
        ],
        cwd=CICERO_REPO,
        env=legacy_build_env,
    )

    if not reuse_pydipcc:
        dipcc_aggregate_error = None

        try:
            run_logged_stage(
                "legacy_make_dipcc",
                [
                    "make",
                    "dipcc",
                ],
                cwd=CICERO_REPO,
                env=legacy_build_env,
            )

        except RuntimeError as exc:
            # The released dipcc CMake graph also contains standalone profiling
            # executables. On newer linkers those can fail even after the
            # required pydipcc shared module has completed successfully.
            dipcc_aggregate_error = str(
                exc
            )

        rebuilt_pydipcc_probe = probe_pydipcc()

        (
            ENGINE_DIR
            / "pydipcc_import_probe.txt"
        ).write_text(
            rebuilt_pydipcc_probe[
                "output"
            ]
        )

        print(
            rebuilt_pydipcc_probe[
                "output"
            ]
        )

        pydipcc_ready = (
            rebuilt_pydipcc_probe[
                "returncode"
            ]
            == 0
            and '"torch": "1.7.1'
            in rebuilt_pydipcc_probe[
                "output"
            ]
            and "cpython-37m"
            in rebuilt_pydipcc_probe[
                "output"
            ]
        )

        if not pydipcc_ready:
            if dipcc_aggregate_error is not None:
                raise RuntimeError(
                    dipcc_aggregate_error
                )

            raise RuntimeError(
                "The dipcc build completed without a usable Python 3.7 "
                "pydipcc extension."
            )

        if dipcc_aggregate_error is not None:
            (
                ENGINE_DIR
                / "dipcc_auxiliary_target_failure.txt"
            ).write_text(
                dipcc_aggregate_error
            )

            print(
                "Required pydipcc extension built and imported successfully; "
                "ignoring failure of auxiliary dipcc executable targets."
            )

    else:
        (
            ENGINE_DIR
            / "pydipcc_import_probe.txt"
        ).write_text(
            existing_pydipcc_probe[
                "output"
            ]
        )

    # Record the generated CMake routing when present.
    dipcc_cache = (
        DIPCC_BUILD_DIR
        / "CMakeCache.txt"
    )

    if dipcc_cache.exists():
        cache_text = dipcc_cache.read_text(
            errors="replace"
        )

        routing_lines = [
            line
            for line in cache_text.splitlines()
            if any(
                key
                in line
                for key in [
                    "Torch_DIR",
                    "PYTHON_EXECUTABLE",
                    "PYTHON_LIBRARY",
                    "Python_EXECUTABLE",
                    "Python_LIBRARY",
                    "CMAKE_PREFIX_PATH",
                ]
            )
        ]

        (
            ENGINE_DIR
            / "dipcc_cmake_routing_after_make.txt"
        ).write_text(
            "\n".join(
                routing_lines
            )
            + "\n"
        )

        host_markers = [
            "/usr/local/lib/python3.12",
            "/usr/local/bin/python",
        ]

        if any(
            marker
            in "\n".join(
                routing_lines
            )
            for marker in host_markers
        ):
            raise RuntimeError(
                "dipcc CMake cache contains notebook-host Python paths; "
                f"see {ENGINE_DIR / 'dipcc_cmake_routing_after_make.txt'}."
            )

    final_stack_probe = capture_command(
        [
            LEGACY_PYTHON,
            "-c",
            (
                "import json,sys,torch,postman; "
                "from fairdiplomacy import pydipcc; "
                "print(json.dumps({"
                "'python':sys.executable,"
                "'torch':torch.__version__,"
                "'cuda_build':torch.version.cuda,"
                "'cuda_available':torch.cuda.is_available(),"
                "'postman':postman.__file__,"
                "'pydipcc':pydipcc.__file__"
                "}))"
            ),
        ],
        cwd=CICERO_REPO,
        env=legacy_build_env,
    )

    (
        RUNTIME_DIR
        / "final_stack_probe.txt"
    ).write_text(
        final_stack_probe[
            "output"
        ]
    )

    print(
        final_stack_probe[
            "output"
        ]
    )

    if final_stack_probe[
        "returncode"
    ] != 0:
        raise RuntimeError(
            "Final source-built Python stack probe failed."
        )

    STACK_MARKER.write_text(
        json.dumps(
            {
                "cicero_commit": CICERO_COMMIT,
                "created_at_utc": datetime.now(
                    timezone.utc
                ).isoformat(),
                "python": str(
                    LEGACY_PYTHON
                ),
                "environment": str(
                    LEGACY_ENV
                ),
                "protoc": {
                    "version": PROTOC_VERSION,
                    "executable": str(
                        PROTOC_EXECUTABLE
                    ),
                },
                "pybind11_build_routing": pybind11_build_routing,
                "pybind11_cmake_dir": str(
                    PYBIND11_CMAKE_DIR
                ),
                "torch_build_routing": torch_build_routing,
                "torch_cmake_prefix": str(
                    TORCH_CMAKE_PREFIX
                ),
                "installer": "pip",
                "cxxflags": legacy_build_env[
                    "CXXFLAGS"
                ],
            },
            indent=2,
        )
    )

(
    ENGINE_DIR
    / "environment_stages.json"
).write_text(
    json.dumps(
        environment_records,
        indent=2,
    )
)

usage = shutil.disk_usage(
    PROJECT_ROOT
)

print({
    "stack_marker": str(
        STACK_MARKER
    ),
    "exists": STACK_MARKER.exists(),
    "free_gib_after_environment": round(
        usage.free
        / (
            1024
            ** 3
        ),
        3,
    ),
})


Reusing existing Postman installation.
python /workspace/latent-reservations/.venvs/cicero_full_agent_py37_source_v5/bin/python
postman /workspace/latent-reservations/vendor/diplomacy_cicero/thirdparty/github/fairinternal/postman/postman/python/postman/__init__.py

Reusing existing Python 3.7 pydipcc build.
{"python": "/workspace/latent-reservations/.venvs/cicero_full_agent_py37_source_v5/bin/python", "torch": "1.7.1+cu110", "pydipcc": "/workspace/latent-reservations/vendor/diplomacy_cicero/fairdiplomacy/pydipcc.cpython-37m-x86_64-linux-gnu.so"}



pip_repo: 0log line [00:00, ?log line/s]

protoc=/workspace/latent-reservations/vendor/cicero_runtime_v5/protoc-3.19.1/bin/protoc
libprotoc 3.19.1
protoc-gen-mypy=/workspace/latent-reservations/.venvs/cicero_full_agent_py37_source_v5/bin/protoc-gen-mypy
pybind11 2.10.4
cmake /workspace/latent-reservations/.venvs/cicero_full_agent_py37_source_v5/lib/python3.7/site-packages/pybind11/share/cmake/pybind11



legacy_make_protos: 0log line [00:00, ?log line/s]

{"python": "/workspace/latent-reservations/.venvs/cicero_full_agent_py37_source_v5/bin/python", "torch": "1.7.1+cu110", "cuda_build": "11.0", "cuda_available": true, "postman": "/workspace/latent-reservations/vendor/diplomacy_cicero/thirdparty/github/fairinternal/postman/postman/python/postman/__init__.py", "pydipcc": "/workspace/latent-reservations/vendor/diplomacy_cicero/fairdiplomacy/pydipcc.cpython-37m-x86_64-linux-gnu.so"}

{'stack_marker': '/workspace/latent-reservations/.venvs/cicero_full_agent_py37_source_v5/.latent_reservations_cicero_stack_v5_7', 'exists': True, 'free_gib_after_environment': 103126.709}


## 9. Legacy GPU and import probes


In [11]:
legacy_probe_code = r"""
import json
import sys
import torch

result = {
    "python": sys.version,
    "python_executable": sys.executable,
    "torch": torch.__version__,
    "cuda_build": torch.version.cuda,
    "cuda_available": torch.cuda.is_available(),
}

if torch.cuda.is_available():
    result["device_count"] = torch.cuda.device_count()
    result["device_name_0"] = torch.cuda.get_device_name(0)

print(json.dumps(result))
"""

legacy_gpu_probe = capture_command(
    [
        str(
            LEGACY_PYTHON
        ),
        "-c",
        legacy_probe_code,
    ],
    cwd=CICERO_REPO,
    env=legacy_build_env,
)

(
    RUNTIME_DIR
    / "legacy_gpu_probe.txt"
).write_text(
    legacy_gpu_probe[
        "output"
    ]
)


# Probe each important import independently so a failure identifies the
# exact module instead of collapsing into a generic import error.
legacy_import_code = r"""
import importlib
import json
import sys
import traceback

result = {
    "python": sys.executable,
    "imports": {},
}

imports = [
    "fairdiplomacy",
    "fairdiplomacy.pydipcc",
    "fairdiplomacy.agents.bqre1p_agent",
]

failed = False

for module_name in imports:
    try:
        module = importlib.import_module(
            module_name
        )

        result["imports"][
            module_name
        ] = {
            "ok": True,
            "file": getattr(
                module,
                "__file__",
                None,
            ),
        }

    except Exception as exc:
        failed = True

        result["imports"][
            module_name
        ] = {
            "ok": False,
            "error_type": type(
                exc
            ).__name__,
            "error": str(
                exc
            ),
            "traceback": traceback.format_exc(),
        }

if not failed:
    from fairdiplomacy import pydipcc
    from fairdiplomacy.agents.bqre1p_agent import (
        BQRE1PAgent,
    )

    result[
        "pydipcc_file"
    ] = pydipcc.__file__

    result[
        "BQRE1PAgent"
    ] = str(
        BQRE1PAgent
    )

print(
    json.dumps(
        result,
        indent=2,
    )
)

if failed:
    sys.exit(
        1
    )
"""

legacy_import_probe = capture_command(
    [
        str(
            LEGACY_PYTHON
        ),
        "-c",
        legacy_import_code,
    ],
    cwd=CICERO_REPO,
    env=legacy_build_env,
)

(
    RUNTIME_DIR
    / "legacy_import_probe.txt"
).write_text(
    legacy_import_probe[
        "output"
    ]
)

print({
    "gpu_probe_returncode": legacy_gpu_probe[
        "returncode"
    ],
    "import_probe_returncode": legacy_import_probe[
        "returncode"
    ],
})

print(
    legacy_gpu_probe[
        "output"
    ]
)

print(
    legacy_import_probe[
        "output"
    ]
)

if legacy_gpu_probe[
    "returncode"
] != 0:
    raise RuntimeError(
        "Python 3.7 GPU probe failed.\n\n"
        + legacy_gpu_probe[
            "output"
        ]
    )

if legacy_import_probe[
    "returncode"
] != 0:
    raise RuntimeError(
        "Full-agent import probe failed. "
        "The diagnostic above identifies the exact failing module.\n\n"
        + legacy_import_probe[
            "output"
        ]
    )

{'gpu_probe_returncode': 0, 'import_probe_returncode': 0}
{"python": "3.7.17 (default, Aug 15 2026, 10:46:23) \n[GCC 13.3.0]", "python_executable": "/workspace/latent-reservations/.venvs/cicero_full_agent_py37_source_v5/bin/python", "torch": "1.7.1+cu110", "cuda_build": "11.0", "cuda_available": true, "device_count": 1, "device_name_0": "NVIDIA L40S"}

{
  "python": "/workspace/latent-reservations/.venvs/cicero_full_agent_py37_source_v5/bin/python",
  "imports": {
    "fairdiplomacy": {
      "ok": true,
      "file": "/workspace/latent-reservations/vendor/diplomacy_cicero/fairdiplomacy/__init__.py"
    },
    "fairdiplomacy.pydipcc": {
      "ok": true,
      "file": "/workspace/latent-reservations/vendor/diplomacy_cicero/fairdiplomacy/pydipcc.cpython-37m-x86_64-linux-gnu.so"
    },
    "fairdiplomacy.agents.bqre1p_agent": {
      "ok": true,
      "file": "/workspace/latent-reservations/vendor/diplomacy_cicero/fairdiplomacy/agents/bqre1p_agent.py"
    }
  },
  "pydipcc_file": "/works

## 10. Verify `run.py` and released config parsing


In [12]:
run_help = capture_command(
    [
        str(
            LEGACY_PYTHON
        ),
        "run.py",
        "--help",
    ],
    cwd=CICERO_REPO,
    env=legacy_build_env,
)

(
    RUNTIME_DIR
    / "run_help.txt"
).write_text(
    run_help[
        "output"
    ]
)

if run_help[
    "returncode"
] != 0:
    raise RuntimeError(
        "Released run.py entry point failed."
    )

print(
    run_help[
        "output"
    ][
        :5000
    ]
)


usage: run.py [-h] -c CFG [--adhoc] [--force]
              [--mode {gentle_start,start_restart,start_continue,restart,dryrun}]
              [--checkpoint CHECKPOINT]
              [--exp_id_pattern_override EXP_ID_PATTERN_OVERRIDE] [--out OUT]
              [--print] [--print-flat] [--print-flat-all]
              [--log-level {ERROR,WARN,INFO,DEBUG}]

optional arguments:
  -h, --help            show this help message and exit
  -c CFG, --cfg CFG
  --adhoc
  --force               Do not ask confirmation in restart mode (default:
                        False)
  --mode {gentle_start,start_restart,start_continue,restart,dryrun}
                        See heyhi/util.py for mode definitions. (default:
                        None)
  --checkpoint CHECKPOINT
                        Directory to automatically copy the code to keep an
                        archive of it. For remote slurm jobs, will also run
                        the code from there so that editing the local repo
       

## 11. Runtime planner API inventory in the legacy process


In [13]:
runtime_inventory_script = (
    RUNTIME_DIR
    / "planner_runtime_inventory.py"
)

runtime_inventory_script.write_text(
    r"""
import inspect
import json

from fairdiplomacy.agents import bqre1p_agent
from fairdiplomacy.agents import br_corr_bilateral_search

classes = {}
for module in [bqre1p_agent, br_corr_bilateral_search]:
    for name, value in vars(module).items():
        if inspect.isclass(value):
            classes[f"{module.__name__}.{name}"] = {
                "module": value.__module__,
                "mro": [x.__name__ for x in value.__mro__],
                "methods": sorted(
                    name
                    for name, item in inspect.getmembers(
                        value,
                        predicate=inspect.isfunction,
                    )
                    if not name.startswith("__")
                ),
            }

functions = {}
for module in [bqre1p_agent, br_corr_bilateral_search]:
    for name, value in vars(module).items():
        if inspect.isfunction(value):
            functions[f"{module.__name__}.{name}"] = {
                "signature": str(inspect.signature(value)),
            }

print(
    json.dumps(
        {
            "classes": classes,
            "functions": functions,
        }
    )
)
"""
)

runtime_inventory = capture_command(
    [
        str(
            LEGACY_PYTHON
        ),
        str(
            runtime_inventory_script
        ),
    ],
    cwd=CICERO_REPO,
    env=legacy_build_env,
)

(
    RUNTIME_DIR
    / "planner_runtime_inventory.json"
).write_text(
    runtime_inventory[
        "output"
    ]
)

if runtime_inventory[
    "returncode"
] != 0:
    raise RuntimeError(
        "Planner runtime inventory failed."
    )

runtime_payload = json.loads(
    runtime_inventory[
        "output"
    ]
)

runtime_signal = {
    "BQRE1PAgent_present": any(
        name.endswith(
            ".BQRE1PAgent"
        )
        for name in runtime_payload[
            "classes"
        ]
    ),
    "run_search_method_present": any(
        "run_search"
        in payload[
            "methods"
        ]
        for payload in runtime_payload[
            "classes"
        ].values()
    ),
    "br_corr_module_present": any(
        name.startswith(
            "fairdiplomacy.agents.br_corr_bilateral_search."
        )
        for name in (
            list(
                runtime_payload[
                    "classes"
                ]
            )
            + list(
                runtime_payload[
                    "functions"
                ]
            )
        )
    ),
}

(
    TABLE_DIR
    / "planner_runtime_signals.json"
).write_text(
    json.dumps(
        runtime_signal,
        indent=2,
    )
)

print(
    json.dumps(
        runtime_signal,
        indent=2,
    )
)


{
  "BQRE1PAgent_present": true,
  "run_search_method_present": true,
  "br_corr_module_present": true
}


## 12. Launch the released full agent and stop after native search is observed


In [14]:
COMPARE_LOG = (
    RUNTIME_DIR
    / "full_agent_startup.log"
)

compare_command = [
    str(
        LEGACY_PYTHON
    ),
    "run.py",
    "--adhoc",
    "--cfg",
    "conf/c01_ag_cmp/cmp.prototxt",
    f"Iagent_one={CICERO_AGENT_INCLUDE}",
    "use_shared_agent=1",
    "power_one=TURKEY",
]

process_env = legacy_build_env.copy()

process_env[
    "PYTHONUNBUFFERED"
] = "1"

process = subprocess.Popen(
    compare_command,
    cwd=CICERO_REPO,
    env=process_env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

line_queue = queue.Queue()

def reader_thread():
    assert process.stdout is not None

    for line in process.stdout:
        line_queue.put(
            line
        )

    line_queue.put(
        None
    )

thread = threading.Thread(
    target=reader_thread,
    daemon=True,
)

thread.start()

search_markers = [
    "BEGINNING BQRE run_search",
    "run_search_with_conditional_evs",
]

planner_search_observed = False
process_exited = False
runtime_samples = []

with COMPARE_LOG.open(
    "w",
    encoding="utf-8",
) as log_handle:
    progress = tqdm(
        desc="Launch released CICERO full agent",
        unit="log line",
    )

    last_runtime_sample = 0.0

    while True:
        try:
            item = line_queue.get(
                timeout=1.0
            )
        except queue.Empty:
            item = "__NO_LINE__"

        if item is None:
            process_exited = True
            break

        if item != "__NO_LINE__":
            log_handle.write(
                item
            )

            log_handle.flush()

            progress.update(
                1
            )

            if any(
                marker
                in item
                for marker in search_markers
            ):
                planner_search_observed = True
                break

        now = time.monotonic()

        if (
            now
            - last_runtime_sample
            >= 15.0
        ):
            sample = capture_command(
                [
                    "nvidia-smi",
                    "--query-compute-apps=pid,process_name,used_memory",
                    "--format=csv,noheader,nounits",
                ]
            )

            runtime_samples.append({
                "timestamp_utc": datetime.now(
                    timezone.utc
                ).isoformat(),
                "output": sample[
                    "output"
                ],
            })

            last_runtime_sample = now

        if process.poll() is not None:
            process_exited = True
            break

    progress.close()

if planner_search_observed:
    try:
        process.send_signal(
            2
        )

        process.wait(
            timeout=30
        )
    except Exception:
        process.terminate()

        try:
            process.wait(
                timeout=15
            )
        except Exception:
            process.kill()

            process.wait()
else:
    if process.poll() is None:
        process.terminate()

        try:
            process.wait(
                timeout=15
            )
        except Exception:
            process.kill()

            process.wait()

(
    RUNTIME_DIR
    / "gpu_runtime_samples.json"
).write_text(
    json.dumps(
        runtime_samples,
        indent=2,
    )
)

startup_result = {
    "command": compare_command,
    "planner_search_observed": planner_search_observed,
    "process_exited_before_search": bool(
        process_exited
        and not planner_search_observed
    ),
    "returncode_after_stop": process.returncode,
    "log_path": str(
        COMPARE_LOG
    ),
}

(
    TABLE_DIR
    / "full_agent_startup_result.json"
).write_text(
    json.dumps(
        startup_result,
        indent=2,
    )
)

print(
    json.dumps(
        startup_result,
        indent=2,
    )
)

if not planner_search_observed:
    raise RuntimeError(
        f"Full agent did not reach native BQRE search. Inspect {COMPARE_LOG}"
    )


Launch released CICERO full agent: 0log line [00:00, ?log line/s]

{
  "command": [
    "/workspace/latent-reservations/.venvs/cicero_full_agent_py37_source_v5/bin/python",
    "run.py",
    "--adhoc",
    "--cfg",
    "conf/c01_ag_cmp/cmp.prototxt",
    "Iagent_one=agents/cicero.prototxt",
    "use_shared_agent=1",
    "power_one=TURKEY"
  ],
  "planner_search_observed": true,
  "process_exited_before_search": false,
  "returncode_after_stop": 1,
  "log_path": "/workspace/latent-reservations/notebook_outputs/14_cicero_full_agent_bootstrap_planner_observability/20260815T122832Z/data/runtime/full_agent_startup.log"
}


## 13. Identify serializable planner observables for the next experiment


In [15]:
OBSERVABLE_PATTERNS = {
    "blueprint_policy": re.compile(
        r"\bbp_policy\b"
    ),
    "plausible_orders": re.compile(
        r"\bplausible_orders\b"
    ),
    "action_probabilities": re.compile(
        r"\bpower_action_ps\b|\bptype_power_action_ps\b"
    ),
    "belief_state": re.compile(
        r"\bbelief_state\b"
    ),
    "search_result": re.compile(
        r"BRMResult|BRCorrBilateralSearchResult"
    ),
    "conditional_values": re.compile(
        r"conditional.*value|value.*table",
        flags=re.IGNORECASE,
    ),
}

observable_hits = []

for row in source_hits:
    text = row[
        "text"
    ]

    matched = [
        name
        for name, pattern in OBSERVABLE_PATTERNS.items()
        if pattern.search(
            text
        )
    ]

    if matched:
        observable_hits.append({
            **row,
            "observable_patterns": matched,
        })

observable_presence = {
    name: any(
        name
        in row[
            "observable_patterns"
        ]
        for row in observable_hits
    )
    for name in OBSERVABLE_PATTERNS
}

(
    SOURCE_DIR
    / "planner_observable_hits.json"
).write_text(
    json.dumps(
        observable_hits,
        indent=2,
    )
)

(
    TABLE_DIR
    / "planner_observable_presence.json"
).write_text(
    json.dumps(
        observable_presence,
        indent=2,
    )
)

print(
    json.dumps(
        observable_presence,
        indent=2,
    )
)


{
  "blueprint_policy": true,
  "plausible_orders": true,
  "action_probabilities": true,
  "belief_state": true,
  "search_result": true,
  "conditional_values": true
}


## 14. Readiness freeze


In [16]:
legacy_gpu_available = False

if legacy_gpu_probe[
    "returncode"
] == 0:
    try:
        gpu_payload = json.loads(
            legacy_gpu_probe[
                "output"
            ].strip().splitlines()[
                -1
            ]
        )

        legacy_gpu_available = bool(
            gpu_payload.get(
                "cuda_available"
            )
        )
    except Exception:
        legacy_gpu_available = False

readiness = {
    "frozen_cicero_commit": actual_commit
    == CICERO_COMMIT,
    "released_model_files_present": bool(
        after_models
    ),
    "legacy_python37_present": LEGACY_PYTHON.exists(),
    "standard_virtual_environment_present": (
        LEGACY_ENV
        / "pyvenv.cfg"
    ).exists(),
    "pip_stack_marker_present": STACK_MARKER.exists(),
    "legacy_full_agent_imports_pass": legacy_import_probe[
        "returncode"
    ]
    == 0,
    "legacy_gpu_visible": legacy_gpu_available,
    "run_py_entrypoint_pass": run_help[
        "returncode"
    ]
    == 0,
    "source_BQRE1PAgent_found": required_source_signals[
        "BQRE1PAgent"
    ],
    "source_run_search_found": required_source_signals[
        "run_search"
    ],
    "runtime_BQRE1PAgent_found": runtime_signal[
        "BQRE1PAgent_present"
    ],
    "runtime_run_search_found": runtime_signal[
        "run_search_method_present"
    ],
    "full_agent_reached_native_search": planner_search_observed,
    "planner_observable_policy_found": (
        observable_presence[
            "blueprint_policy"
        ]
        and observable_presence[
            "plausible_orders"
        ]
        and observable_presence[
            "action_probabilities"
        ]
    ),
}

readiness[
    "ready_for_notebook15_cicero_planner_capture"
] = bool(
    all(
        readiness.values()
    )
)

(
    TABLE_DIR
    / "readiness.json"
).write_text(
    json.dumps(
        readiness,
        indent=2,
    )
)

print(
    json.dumps(
        readiness,
        indent=2,
    )
)


{
  "frozen_cicero_commit": true,
  "released_model_files_present": true,
  "legacy_python37_present": true,
  "standard_virtual_environment_present": true,
  "pip_stack_marker_present": true,
  "legacy_full_agent_imports_pass": true,
  "legacy_gpu_visible": true,
  "run_py_entrypoint_pass": true,
  "source_BQRE1PAgent_found": true,
  "source_run_search_found": true,
  "runtime_BQRE1PAgent_found": true,
  "runtime_run_search_found": true,
  "full_agent_reached_native_search": true,
  "planner_observable_policy_found": true,
  "ready_for_notebook15_cicero_planner_capture": true
}


## 15. Final summary and manifest


In [17]:
result_summary = {
    "notebook_build": NOTEBOOK_BUILD,
    "run_id": RUN_ID,
    "cicero_commit": CICERO_COMMIT,
    "purpose": (
        "Bootstrap the released full CICERO agent and map native strategic planner observables."
    ),
    "agent_config": {
        "compare_task_config": str(COMPARE_AGENTS_CONFIG),
        "cicero_include": CICERO_AGENT_INCLUDE,
        "resolved_cicero_agent_config": str(
            cicero_agent_config.relative_to(
                CICERO_REPO
            )
        ),
    },
    "scientific_status": (
        "Engineering/mechanistic observability validation only; no H1 estimate."
    ),
    "model_inventory": {
        "files": int(
            len(
                after_models
            )
        ),
        "total_bytes": int(
            sum(
                row[
                    "size_bytes"
                ]
                for row in after_models
            )
        ),
    },
    "source_signals": required_source_signals,
    "runtime_signals": runtime_signal,
    "observable_presence": observable_presence,
    "startup_result": startup_result,
    "readiness": readiness,
}

(
    TABLE_DIR
    / "result_summary.json"
).write_text(
    json.dumps(
        result_summary,
        indent=2,
    )
)

manifest = {
    "notebook_build": NOTEBOOK_BUILD,
    "run_id": RUN_ID,
    "run_output_dir": str(
        RUN_DIR
    ),
    "host_inventory": str(
        ENGINE_DIR
        / "host_inventory.json"
    ),
    "selected_model_subset": str(
        TABLE_DIR
        / "selected_model_subset.json"
    ),
    "selected_model_remote_sizes": str(
        TABLE_DIR
        / "selected_model_remote_sizes.json"
    ),
    "storage_plan": str(
        TABLE_DIR
        / "storage_plan.json"
    ),
    "model_inventory": str(
        ENGINE_DIR
        / "model_inventory.json"
    ),
    "source_runtime_bootstrap": str(
        ENGINE_DIR
        / "source_runtime_bootstrap.json"
    ),
    "compatibility_flags": str(
        ENGINE_DIR
        / "compatibility_flags.json"
    ),
    "environment_stages": str(
        ENGINE_DIR
        / "environment_stages.json"
    ),
    "legacy_gpu_probe": str(
        RUNTIME_DIR
        / "legacy_gpu_probe.txt"
    ),
    "legacy_import_probe": str(
        RUNTIME_DIR
        / "legacy_import_probe.txt"
    ),
    "run_help": str(
        RUNTIME_DIR
        / "run_help.txt"
    ),
    "planner_source_hits": str(
        SOURCE_DIR
        / "planner_source_hits.json"
    ),
    "planner_runtime_inventory": str(
        RUNTIME_DIR
        / "planner_runtime_inventory.json"
    ),
    "full_agent_startup_log": str(
        COMPARE_LOG
    ),
    "gpu_runtime_samples": str(
        RUNTIME_DIR
        / "gpu_runtime_samples.json"
    ),
    "planner_observable_hits": str(
        SOURCE_DIR
        / "planner_observable_hits.json"
    ),
    "readiness": str(
        TABLE_DIR
        / "readiness.json"
    ),
    "result_summary": str(
        TABLE_DIR
        / "result_summary.json"
    ),
}

(
    MANIFEST_DIR
    / "notebook14_manifest.json"
).write_text(
    json.dumps(
        manifest,
        indent=2,
    )
)

print(
    json.dumps(
        result_summary,
        indent=2,
    )
)

print()
print("Notebook 14 complete.")
print(
    f"Result summary: {TABLE_DIR / 'result_summary.json'}"
)
print(
    f"Run directory: {RUN_DIR}"
)


{
  "notebook_build": "latent-reservations-notebook14-cicero-full-agent-bootstrap-v5.7-source-py37-venv",
  "run_id": "20260815T122832Z",
  "cicero_commit": "e85afeddb34f5b7c1ea0827203b425a0f7e68ead",
  "purpose": "Bootstrap the released full CICERO agent and map native strategic planner observables.",
  "agent_config": {
    "compare_task_config": "/workspace/latent-reservations/vendor/diplomacy_cicero/conf/c01_ag_cmp/cmp.prototxt",
    "cicero_include": "agents/cicero.prototxt",
    "resolved_cicero_agent_config": "conf/common/agents/cicero.prototxt"
  },
  "scientific_status": "Engineering/mechanistic observability validation only; no H1 estimate.",
  "model_inventory": {
    "files": 67,
    "total_bytes": 33463639523
  },
  "source_signals": {
    "BQRE1PAgent": true,
    "run_search": true,
    "blueprint_policy": true,
    "plausible_orders": true,
    "action_probabilities": true,
    "belief_state": true,
    "bilateral": true
  },
  "runtime_signals": {
    "BQRE1PAgent_prese